<a href="https://colab.research.google.com/github/mardocheeogecime-gif/Rede_Colabora-o_Acad-mica_ECI_UFMG/blob/main/Lab_Exploring_intra_field_collaboration_Brazilian_Information_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# PARTE 0 — Setup, Drive, Run Manager, Logs
# ==========================================
import os, sys, json, time, math, re, random, platform, hashlib
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, Tuple, List

import numpy as np
import pandas as pd

# ---- Colab/Drive (monta e cria pastas) ----
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ---- Reprodutibilidade ----
def set_global_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

# ---- Normalização de nomes: token-sort robusto ----
def normalize_name_token_sort(s: Any) -> str:
    import unicodedata
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 2]
    toks.sort()
    return " ".join(toks)

# ---- Hash curto para identificar arquivos/runs ----
def short_hash_bytes(b: bytes, n: int = 10) -> str:
    return hashlib.sha256(b).hexdigest()[:n]

def short_hash_file(path: str, n: int = 10) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n]

# ---- Impressão estilo-artigo (tabelas limpas) ----
def ascii_title(s: str) -> str:
    line = "=" * max(len(s), 10)
    return f"{s}\n{line}"

def print_table(title: str, df: pd.DataFrame, fmt: Optional[Dict[str, Any]] = None, pad: int = 2) -> None:
    print(ascii_title(title))
    if df.empty:
        print("(vazio)\n")
        return
    cols = list(df.columns)
    rows = df.values.tolist()

    def fmt_val(col, v):
        if fmt and col in fmt:
            return fmt[col](v)
        if isinstance(v, (float, np.floating)):
            return f"{v:.3f}".rstrip("0").rstrip(".")
        return str(v)

    col_widths = []
    for i, c in enumerate(cols):
        mx = len(str(c))
        for r in rows:
            mx = max(mx, len(fmt_val(c, r[i])))
        col_widths.append(mx)

    header = " ".join(str(c).ljust(col_widths[i] + pad) for i, c in enumerate(cols))
    print(header)
    print("-" * len(header))
    for r in rows:
        print(" ".join(fmt_val(cols[i], r[i]).ljust(col_widths[i] + pad) for i in range(len(cols))))
    print()

# ---- Run Manager: cada execução gera uma pasta versionada ----
@dataclass
class RunConfig:
    project_root: str
    run_root: str
    seed: int = 42
    note: str = ""
    created_at: str = ""

def make_run_folders(project_root: str, note: str = "", seed: int = 42) -> RunConfig:
    os.makedirs(project_root, exist_ok=True)
    runs_dir = os.path.join(project_root, "runs")
    os.makedirs(runs_dir, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_root = os.path.join(runs_dir, f"run_{ts}")
    os.makedirs(run_root, exist_ok=True)

    # subpastas padronizadas
    for sub in ["input", "intermediate", "tables", "figures", "models", "logs", "artifacts"]:
        os.makedirs(os.path.join(run_root, sub), exist_ok=True)

    cfg = RunConfig(
        project_root=project_root,
        run_root=run_root,
        seed=seed,
        note=note,
        created_at=datetime.now().isoformat(timespec="seconds"),
    )
    return cfg

def save_json(obj: Dict[str, Any], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def save_text(s: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(s)

def snapshot_environment(cfg: RunConfig) -> None:
    env = {
        "created_at": cfg.created_at,
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "in_colab": IN_COLAB,
        "seed": cfg.seed,
        "note": cfg.note,
        "cwd": os.getcwd(),
    }
    save_json(env, os.path.join(cfg.run_root, "logs", "environment.json"))

# ---- Defina onde salvar no Drive ----
# Ajuste se quiser outro caminho no seu Drive.
DEFAULT_PROJECT_ROOT = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"

# ---- Cria run ----
cfg = make_run_folders(DEFAULT_PROJECT_ROOT, note="lab_v2 (upload + persistência + audit)", seed=42)
set_global_seed(cfg.seed)
snapshot_environment(cfg)

print("Run criado em:", cfg.run_root)
print("Pastas:", os.listdir(cfg.run_root))


Mounted at /content/drive
Run criado em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407
Pastas: ['input', 'intermediate', 'tables', 'figures', 'models', 'logs', 'artifacts']


In [ ]:
# ==========================================
# PARTE 1 — Upload + Padronização (CSV/XLSX) + Persistência
# - Modo A: nodes_plus_clusters + edges_beta
# - Modo B: autores_final + producoes_final (+ sucupira opcional) (XLSX) -> gera nodes/edges intermediários
# ==========================================
import os, re, json
import pandas as pd
import numpy as np

# ---------- utils ----------
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def short_hash_file(path: str, n: int = 10) -> str:
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n]

def pick_first_present(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def detect_edge_columns(edges_raw: pd.DataFrame):
    cols = list(edges_raw.columns)
    pairs = [
        ("source","target"), ("src","dst"), ("from","to"),
        ("u","v"), ("i","j"), ("n1","n2"),
        ("node_u","node_v"), ("author_u","author_v"),
        ("author1","author2"), ("a","b"),
    ]
    lowmap = {c.lower(): c for c in cols}
    src = dst = None
    for a,b in pairs:
        if a in lowmap and b in lowmap:
            src, dst = lowmap[a], lowmap[b]
            break
    if src is None:
        nonnum = [c for c in cols if not pd.api.types.is_numeric_dtype(edges_raw[c])]
        if len(nonnum) >= 2:
            src, dst = nonnum[0], nonnum[1]
        else:
            src, dst = cols[0], cols[1]

    # peso
    weight_cands = ["weight","w","edge_weight","count","freq","strength"]
    wcol = None
    for c in cols:
        cl = c.lower()
        if any(tag in cl for tag in weight_cands) and pd.api.types.is_numeric_dtype(edges_raw[c]):
            wcol = c
            break
    if wcol is None:
        nums = [c for c in cols if pd.api.types.is_numeric_dtype(edges_raw[c])]
        wcol = nums[0] if nums else None

    return src, dst, wcol

def normalize_name_token_sort(s):
    import unicodedata
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 2]
    toks.sort()
    return " ".join(toks)

def print_kv(title, d):
    print(title)
    for k,v in d.items():
        print(f" - {k}: {v}")

# ---------- upload ----------
def upload_files_to_run(cfg, expected=None):
    if not IN_COLAB:
        raise RuntimeError("Upload interativo foi desenhado para Google Colab.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Nenhum arquivo foi enviado.")

    saved = {}
    for fname, content in uploaded.items():
        save_path = os.path.join(cfg.run_root, "input", fname)
        with open(save_path, "wb") as f:
            f.write(content)
        saved[fname] = save_path

    meta = []
    for fname, path in saved.items():
        meta.append({
            "filename": fname,
            "saved_path": path,
            "sha10": short_hash_file(path),
            "bytes": os.path.getsize(path),
        })
    save_json({"uploaded": meta}, os.path.join(cfg.run_root, "logs", "upload_manifest.json"))

    if expected:
        missing = [e for e in expected if e not in saved]
        if missing:
            print("⚠️ Atenção: arquivos esperados não encontrados no upload:", missing)
            print("Arquivos enviados:", list(saved.keys()))
    return saved

def find_uploaded_path(saved, preferred_stem):
    """
    preferred_stem: ex 'nodes_plus_clusters' (sem extensão) ou 'nodes_plus_clusters.csv'
    """
    stem = preferred_stem.replace(".csv","").replace(".xlsx","").lower()
    # match exato por stem em filename
    for k,v in saved.items():
        kk = k.lower()
        if kk == stem or kk == stem + ".csv" or kk == stem + ".xlsx":
            return v
    # match por substring
    for k,v in saved.items():
        if stem in k.lower():
            return v
    return None

# ---------- leitura multi-formato ----------
def read_table(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in [".xlsx", ".xls"]:
        # pega primeira aba por padrão
        return pd.read_excel(path)
    raise ValueError(f"Formato não suportado: {ext} ({path})")

# ==========================================
# 1) UPLOAD
# ==========================================
expected_any = [
    "nodes_plus_clusters.csv", "edges_beta.csv",
    "autores_final.xlsx", "producoes_final.xlsx", "banco-sucupira_docente-PPGCI.xlsx"
]
saved = upload_files_to_run(cfg, expected=expected_any)

print("Salvos em:", os.path.join(cfg.run_root, "input"))
print("Arquivos:", list(saved.keys()))

# ==========================================
# 2) DETECTA MODO
# ==========================================
nodes_path = find_uploaded_path(saved, "nodes_plus_clusters")
edges_path = find_uploaded_path(saved, "edges_beta")

autores_path   = find_uploaded_path(saved, "autores_final")
producoes_path = find_uploaded_path(saved, "producoes_final")
sucupira_path  = find_uploaded_path(saved, "banco-sucupira_docente-PPGCI")

mode = None
if nodes_path and edges_path:
    mode = "A_part2_ready"
elif autores_path and producoes_path:
    mode = "B_raw_xlsx"
else:
    raise FileNotFoundError(
        "Não encontrei os arquivos mínimos para rodar.\n"
        "Envie OU (nodes_plus_clusters + edges_beta) OU (autores_final + producoes_final).\n"
        f"Arquivos recebidos: {list(saved.keys())}"
    )

save_json({"mode": mode}, os.path.join(cfg.run_root, "logs", "mode_selected.json"))
print("✅ MODO selecionado:", mode)

# ==========================================
# 3) MODO A — ingestão direta (nodes/edges prontos)
# ==========================================
if mode == "A_part2_ready":
    nodes = read_table(nodes_path)
    edges_raw = read_table(edges_path)

    # nome normalizado do nó
    if "nome_tokensort_norm" not in nodes.columns:
        raw_name_col = pick_first_present(list(nodes.columns), ["NM_DOCENTE","nm_docente","nome","name"])
        if raw_name_col is None:
            raise KeyError(
                "nodes_plus_clusters precisa ter 'nome_tokensort_norm' ou uma coluna de nome bruto.\n"
                f"Colunas encontradas: {list(nodes.columns)}"
            )
        nodes["nome_tokensort_norm"] = nodes[raw_name_col].map(normalize_name_token_sort)

    if "cluster_mod" not in nodes.columns:
        raise KeyError(
            "nodes_plus_clusters precisa conter 'cluster_mod' (rótulo de comunidade).\n"
            f"Colunas encontradas: {list(nodes.columns)}"
        )

    src_col, dst_col, w_col = detect_edge_columns(edges_raw)
    E = pd.DataFrame({
        "u": edges_raw[src_col].astype(str).map(normalize_name_token_sort),
        "v": edges_raw[dst_col].astype(str).map(normalize_name_token_sort),
    })
    E["w"] = edges_raw[w_col] if w_col is not None else 1.0

    doc_set = set(nodes["nome_tokensort_norm"].astype(str))
    E = E[(E["u"].isin(doc_set)) & (E["v"].isin(doc_set))].copy()
    E = E[(E["u"] != "") & (E["v"] != "")]
    E = E.groupby(["u","v"], as_index=False)["w"].sum()

    if E.empty:
        raise ValueError("edges_beta não casou com nomes dos nós após normalização. Verifique colunas e normalização.")

    diag = {
        "mode": mode,
        "nodes_rows": int(nodes.shape[0]),
        "edges_raw_rows": int(edges_raw.shape[0]),
        "edges_matched_rows": int(E.shape[0]),
        "src_col": src_col, "dst_col": dst_col, "w_col": w_col,
        "nodes_unique_norm": int(nodes["nome_tokensort_norm"].nunique()),
    }
    save_json(diag, os.path.join(cfg.run_root, "logs", "ingestion_diagnostics.json"))
    print_kv("Ingestão — Diagnóstico", diag)

    # persistência
    nodes_clean_path = os.path.join(cfg.run_root, "intermediate", "nodes_clean.csv")
    edges_clean_path = os.path.join(cfg.run_root, "intermediate", "edges_clean.csv")
    nodes.to_csv(nodes_clean_path, index=False)
    E.to_csv(edges_clean_path, index=False)

    print("✅ Persistido para reuso:")
    print(" -", nodes_clean_path)
    print(" -", edges_clean_path)

# ==========================================
# 4) MODO B — construir nodes/edges a partir de autores_final + producoes_final (+ sucupira opcional)
# ==========================================
if mode == "B_raw_xlsx":
    autores = read_table(autores_path)
    producoes = read_table(producoes_path)
    sucupira = read_table(sucupira_path) if sucupira_path else None

    # ---- detectar colunas de nomes ----
    # autores_final: esperamos algo como 'nome_pesquisado' OU 'nome_tokensort_norm' OU algo equivalente
    a_name = pick_first_present(list(autores.columns), ["nome_pesquisado","nome_tokensort_norm","nome","NM_DOCENTE","author","name"])
    if a_name is None:
        raise KeyError(
            "autores_final: não encontrei coluna de nome.\n"
            f"Colunas encontradas: {list(autores.columns)}"
        )

    if "nome_tokensort_norm" not in autores.columns:
        autores["nome_tokensort_norm"] = autores[a_name].map(normalize_name_token_sort)

    # ---- mapeamento institucional (se existir) ----
    # sua memória: autores_final tem 'instituicao_original' e 'nome_pesquisado'
    inst_col = pick_first_present(list(autores.columns), ["instituicao_original","SG_ENTIDADE_ENSINO","NM_ENTIDADE_ENSINO","instituicao","ies"])
    if inst_col is None:
        # ok: segue sem IES no nodes; entra depois
        inst_col = None

    # ---- producoes_final: precisamos dos autores por publicação ----
    # tentamos achar:
    # - um ID da produção (work_id / id / doi / titulo)
    # - uma coluna com lista de autores (string) OU colunas autor1/autor2...
    p_id = pick_first_present(list(producoes.columns), ["work_id","id","openalex_id","doi","titulo","title"])
    if p_id is None:
        raise KeyError(
            "producoes_final: não encontrei uma coluna identificadora da produção (id/doi/titulo).\n"
            f"Colunas encontradas: {list(producoes.columns)}"
        )

    # coluna de autores: tentamos vários nomes comuns
    p_auth = pick_first_present(list(producoes.columns), ["autores","authors","author_list","lista_autores","nm_docente","docentes"])
    # fallback: tenta colunas autor_1, autor_2...
    author_cols = [c for c in producoes.columns if re.search(r"(autor|author)\D*\d+", c.lower())]

    if p_auth is None and not author_cols:
        raise KeyError(
            "producoes_final: não encontrei coluna com autores (ex: autores/authors) nem colunas autor1/autor2.\n"
            f"Colunas encontradas: {list(producoes.columns)}"
        )

    # ---- construir lista de autores por produção ----
    # Estratégia:
    # - se existir uma coluna string (p_auth): split por ; , | e normaliza
    # - senão usa author_cols: junta não-nulos
    def parse_authors_row(row):
        names = []
        if p_auth is not None:
            raw = row[p_auth]
            if pd.isna(raw):
                return []
            raw = str(raw)
            parts = re.split(r"[;,\|/]+", raw)
            names = [normalize_name_token_sort(x) for x in parts if str(x).strip()]
        else:
            tmp = []
            for c in author_cols:
                v = row[c]
                if pd.notna(v) and str(v).strip():
                    tmp.append(normalize_name_token_sort(v))
            names = tmp
        # remove vazios e dup
        names = [n for n in names if n]
        return sorted(set(names))

    # ---- filtro intra-PPGCI: só autores que estão no autores_final ----
    ppgci_set = set(autores["nome_tokensort_norm"].astype(str))
    # cria uma lista de autores ppgci por produção
    work_authors = []
    for _, row in producoes.iterrows():
        wid = row[p_id]
        auths = parse_authors_row(row)
        auths_ppgci = [a for a in auths if a in ppgci_set]
        if len(auths_ppgci) >= 2:
            work_authors.append((wid, auths_ppgci))

    if not work_authors:
        raise ValueError(
            "Nenhuma produção com >=2 autores PPGCI foi encontrada após o match.\n"
            "Isso geralmente indica divergência de normalização/nomenclatura entre autores_final e producoes_final."
        )

    # ---- gera arestas (pares) por produção ----
    from itertools import combinations
    edge_counts = {}
    for wid, auths_ppgci in work_authors:
        for u, v in combinations(sorted(auths_ppgci), 2):
            key = (u, v)
            edge_counts[key] = edge_counts.get(key, 0) + 1

    E = pd.DataFrame([{"u": u, "v": v, "w": w} for (u, v), w in edge_counts.items()])

    # ---- nodes base ----
    nodes = pd.DataFrame({"nome_tokensort_norm": sorted(ppgci_set)})
    # anexa instituição se disponível
    if inst_col is not None:
        inst_map = dict(zip(autores["nome_tokensort_norm"], autores[inst_col].astype(str)))
        nodes["instituicao_original"] = nodes["nome_tokensort_norm"].map(inst_map)

    # sucupira opcional: tenta mapear sigla/nome da instituição se existir match textual
    if sucupira is not None:
        # colunas conhecidas do seu banco: NM_ENTIDADE_ENSINO / SG_ENTIDADE_ENSINO
        nm_ies = pick_first_present(list(sucupira.columns), ["NM_ENTIDADE_ENSINO"])
        sg_ies = pick_first_present(list(sucupira.columns), ["SG_ENTIDADE_ENSINO"])
        if nm_ies and sg_ies and "instituicao_original" in nodes.columns:
            # normaliza nm_ies para mapear por aproximação simples (mesma string)
            sucupira["_ies_norm"] = sucupira[nm_ies].astype(str).str.strip().str.lower()
            nodes["_ies_norm"] = nodes["instituicao_original"].astype(str).str.strip().str.lower()
            map_sigla = dict(zip(sucupira["_ies_norm"], sucupira[sg_ies].astype(str)))
            nodes["SG_ENTIDADE_ENSINO"] = nodes["_ies_norm"].map(map_sigla)
            nodes.drop(columns=["_ies_norm"], inplace=True)

    # ---- diagnósticos ----
    diag = {
        "mode": mode,
        "autores_rows": int(autores.shape[0]),
        "producoes_rows": int(producoes.shape[0]),
        "ppgci_unique_authors": int(len(ppgci_set)),
        "works_with_>=2_ppgci_authors": int(len(work_authors)),
        "edges_generated": int(E.shape[0]),
        "weight_definition": "w = #coauthored_works (intra-PPGCI only)",
        "autores_name_col_used": a_name,
        "autores_inst_col_used": inst_col,
        "producoes_id_col_used": p_id,
        "producoes_auth_col_used": p_auth if p_auth else ("multi_cols:" + ",".join(author_cols[:5]) + ("..." if len(author_cols)>5 else "")),
        "sucupira_used": bool(sucupira is not None),
    }
    save_json(diag, os.path.join(cfg.run_root, "logs", "ingestion_diagnostics.json"))
    print_kv("Ingestão — Diagnóstico (RAW XLSX)", diag)

    # ---- persistência ----
    nodes_clean_path = os.path.join(cfg.run_root, "intermediate", "nodes_clean.csv")
    edges_clean_path = os.path.join(cfg.run_root, "intermediate", "edges_clean.csv")
    nodes.to_csv(nodes_clean_path, index=False)
    E.to_csv(edges_clean_path, index=False)

    print("✅ Gerados e persistidos (para Partes 2+):")
    print(" -", nodes_clean_path)
    print(" -", edges_clean_path)

    # também salva uma cópia do "works_with_authors" para auditoria
    audit_path = os.path.join(cfg.run_root, "intermediate", "works_ppgci_pairs_audit.csv")
    audit_df = pd.DataFrame([{"work_id": wid, "authors_ppgci": ";".join(auths)} for wid, auths in work_authors])
    audit_df.to_csv(audit_path, index=False)
    print(" -", audit_path)

print("\n✅ PARTE 1 concluída. Próximo passo: PARTE 2 (construção do grafo + giant component + métricas).")


Saving banco-sucupira_docente-PPGCI.xlsx to banco-sucupira_docente-PPGCI (1).xlsx
Saving autores_final.xlsx to autores_final (1).xlsx
Saving producoes_final.xlsx to producoes_final (1).xlsx
⚠️ Atenção: arquivos esperados não encontrados no upload: ['nodes_plus_clusters.csv', 'edges_beta.csv', 'autores_final.xlsx', 'producoes_final.xlsx', 'banco-sucupira_docente-PPGCI.xlsx']
Arquivos enviados: ['banco-sucupira_docente-PPGCI (1).xlsx', 'autores_final (1).xlsx', 'producoes_final (1).xlsx']
Salvos em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/input
Arquivos: ['banco-sucupira_docente-PPGCI (1).xlsx', 'autores_final (1).xlsx', 'producoes_final (1).xlsx']
✅ MODO selecionado: B_raw_xlsx
Ingestão — Diagnóstico (RAW XLSX)
 - mode: B_raw_xlsx
 - autores_rows: 543
 - producoes_rows: 38138
 - ppgci_unique_authors: 510
 - works_with_>=2_ppgci_authors: 8122
 - edges_generated: 1609
 - weight_definition: w = #coauthored_works (intra-PPGCI only)
 - autores_name_col_used: 

In [ ]:
# ==========================================
# PARTE 2 — Grafo + Disparity Filter + β sweep + Comunidades + Auditoria
# ==========================================
import os, re, json, math, time
import numpy as np
import pandas as pd
import networkx as nx

# ---------- helpers ----------
def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def normalize_name_token_sort(s):
    import unicodedata
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 2]
    toks.sort()
    return " ".join(toks)

def find_latest_run(root_base):
    if not os.path.exists(root_base):
        return None
    runs = [d for d in os.listdir(root_base) if d.startswith("run_")]
    if not runs:
        return None
    runs = sorted(runs)
    return os.path.join(root_base, runs[-1])

def pick_first_present(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

# ---------- sociotech metadata detection ----------
def detect_sociotech_cols(nodes_df):
    cols = list(nodes_df.columns)
    cand = {
        "UF": ["SG_UF_PROGRAMA","uf","SG_UF","estado","state"],
        "IES": ["SG_ENTIDADE_ENSINO","NM_ENTIDADE_ENSINO","ies","instituicao","instituicao_original","institution"],
        "PROGRAMA": ["NM_PROGRAMA_IES","programa","nm_programa","program"],
        "NOTA": ["NOTA_PROGRAMA_IES","nota_programa","nota","capes_grade","grade"],
    }
    out = {}
    for k, cands in cand.items():
        out[k] = pick_first_present(cols, cands)
    return out

# ---------- Cramér's V ----------
def cramers_v(x, y):
    import pandas as pd
    from scipy.stats import chi2_contingency
    tbl = pd.crosstab(pd.Series(x, dtype="category"), pd.Series(y, dtype="category"))
    if tbl.size == 0:
        return np.nan, np.nan, (tbl.shape[0], tbl.shape[1])
    chi2, p, _, _ = chi2_contingency(tbl, correction=False)
    n = tbl.to_numpy().sum()
    r, c = tbl.shape
    denom = n * (min(r, c) - 1)
    if denom <= 0:
        return np.nan, float(p), (r, c)
    V = math.sqrt(chi2 / denom)
    return float(V), float(p), (r, c)

# ---------- Disparity Filter (undirected, weighted) ----------
def disparity_filter_backbone(G, beta=0.5, weight="weight", or_rule=True):
    """
    Retém aresta (i,j) se alpha_ij < beta (OR rule por endpoint).
    Forma fechada: alpha_ij = (1 - p_ij)^(k_i - 1), para k_i > 1
    p_ij = w_ij / s_i
    """
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))

    degree = {n: G.degree(n) for n in G.nodes()}
    strength = {}
    for n in G.nodes():
        s = 0.0
        for _, _, d in G.edges(n, data=True):
            s += float(d.get(weight, 1.0))
        strength[n] = s

    def alpha(i, w_ij):
        k = degree[i]
        if k <= 1:
            return 0.0  # nó folha: mantém arestas
        s = strength[i]
        if s <= 0:
            return 1.0
        p = max(min(w_ij / s, 1.0), 0.0)
        return (1.0 - p) ** (k - 1)

    for u, v, d in G.edges(data=True):
        w = float(d.get(weight, 1.0))
        a_uv = alpha(u, w)
        a_vu = alpha(v, w)
        keep = (a_uv < beta) or (a_vu < beta) if or_rule else ((a_uv < beta) and (a_vu < beta))
        if keep:
            H.add_edge(u, v, **{weight: w})

    return H

# ---------- Graph-native metrics ----------
def graph_metrics(G, communities, weight="weight"):
    from networkx.algorithms.community.quality import modularity

    Q = modularity(G, communities, weight=weight) if G.number_of_edges() > 0 else float("nan")

    total_w = sum(float(d.get(weight, 1.0)) for _,_,d in G.edges(data=True))
    if total_w <= 0:
        coverage = float("nan")
    else:
        w_in = 0.0
        for S in communities:
            SG = G.subgraph(S)
            w_in += sum(float(d.get(weight, 1.0)) for _,_,d in SG.edges(data=True))
        coverage = w_in / total_w

    strength = {n: 0.0 for n in G.nodes()}
    for u, v, d in G.edges(data=True):
        w = float(d.get(weight, 1.0))
        strength[u] += w
        strength[v] += w
    total_vol = sum(strength.values())

    def cut_weight(S):
        S = set(S)
        cw = 0.0
        for u, v, d in G.edges(data=True):
            if (u in S) ^ (v in S):
                cw += float(d.get(weight, 1.0))
        return cw

    rows = []
    for cid, S in enumerate(communities):
        S = set(S)
        vol = sum(strength[n] for n in S)
        cw = cut_weight(S)
        denom = min(vol, total_vol - vol)
        phi = (cw / denom) if denom and denom > 0 else 0.0
        rows.append([cid, len(S), vol, cw, phi])
    cond_df = pd.DataFrame(rows, columns=["community","size","volume","cut","conductance"])
    cond_mean = float(cond_df["conductance"].mean()) if len(cond_df) else float("nan")
    cond_wmean = float(np.average(cond_df["conductance"], weights=np.clip(cond_df["volume"].values, 1e-12, None))) if len(cond_df) else float("nan")

    # Asymptotic Surprise (aprox. não-ponderada)
    n = G.number_of_nodes()
    m = G.number_of_edges()
    if n < 2 or m < 1:
        S_asym = float("nan")
    else:
        M = n*(n-1)//2
        k_in = 0
        M_in = 0
        for S in communities:
            S = set(S)
            k_in += G.subgraph(S).number_of_edges()
            s = len(S)
            M_in += s*(s-1)//2
        eps = 1e-15
        p_in  = max(min(k_in / max(m, 1), 1-eps), eps)
        p_exp = max(min(M_in / max(M, 1), 1-eps), eps)
        S_asym = m * ( p_in*math.log(p_in/p_exp) + (1-p_in)*math.log((1-p_in)/(1-p_exp)) )

    return {
        "modularity_Q": float(Q),
        "coverage": float(coverage),
        "conductance_mean": float(cond_mean),
        "conductance_weighted_mean": float(cond_wmean),
        "asymptotic_surprise": float(S_asym),
        "conductance_df": cond_df
    }

# ---------- robust Louvain import (python-louvain) ----------
def import_python_louvain():
    """
    Evita shadowing do módulo 'community' (que pode NÃO ser python-louvain).
    Preferir: import community.community_louvain as community_louvain
    """
    try:
        import community.community_louvain as community_louvain
        if not hasattr(community_louvain, "best_partition"):
            raise ImportError("community.community_louvain sem best_partition (shadowing).")
        return community_louvain
    except Exception:
        import sys, subprocess, importlib
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-louvain"])
        # reimport após instalar
        community_louvain = importlib.import_module("community.community_louvain")
        if not hasattr(community_louvain, "best_partition"):
            raise ImportError("Falha ao importar python-louvain (best_partition ausente).")
        return community_louvain

# ---------- Community detection (Leiden preferred; fallback Louvain) ----------
def best_partition(G, weight="weight", seed=42, prefer="leiden"):
    """
    Retorna (labels_map, algo)
    labels_map: node -> cluster_id
    """
    if G.number_of_edges() == 0:
        return {u: i for i, u in enumerate(G.nodes())}, "degenerate"

    # Leiden (se disponível)
    if prefer == "leiden":
        try:
            import igraph as ig
            import leidenalg as la

            nodes = list(G.nodes())
            idx = {n:i for i,n in enumerate(nodes)}
            edges = [(idx[u], idx[v]) for u,v in G.edges()]
            weights = [float(G[u][v].get(weight, 1.0)) for u,v in G.edges()]

            g = ig.Graph(n=len(nodes), edges=edges, directed=False)
            g.es["weight"] = weights
            part = la.find_partition(
                g,
                la.RBConfigurationVertexPartition,
                weights="weight",
                resolution_parameter=1.0,
                seed=seed
            )
            labels = {nodes[i]: int(part.membership[i]) for i in range(len(nodes))}
            return labels, "leiden"
        except Exception:
            pass

    # Louvain fallback (robusto)
    community_louvain = import_python_louvain()
    labels = community_louvain.best_partition(G, weight=weight, random_state=seed)
    labels = {k: int(v) for k,v in labels.items()}
    return labels, "louvain"

def communities_from_labels(labels_map):
    from collections import defaultdict
    groups = defaultdict(list)
    for n, c in labels_map.items():
        groups[c].append(n)
    return [set(groups[k]) for k in sorted(groups.keys())]

# ---------- β selection: Pareto + parcimônia ----------
def pareto_front(df, maximize_cols):
    X = df[maximize_cols].to_numpy(float)
    n = X.shape[0]
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        if not is_pareto[i]:
            continue
        dominates_i = np.all(X >= X[i], axis=1) & np.any(X > X[i], axis=1)
        if np.any(dominates_i):
            is_pareto[i] = False
    return is_pareto

def select_beta_star(audit_df):
    maximize = ["modularity_Q","V_UF","V_IES","H_norm","coverage"]
    use_cols = [c for c in maximize if c in audit_df.columns and audit_df[c].notna().any()]
    if not use_cols:
        best = audit_df.sort_values(["modularity_Q","beta"], ascending=[False, True]).iloc[0]
        return float(best["beta"]), {"rule":"fallback_maxQ_then_minbeta", "used_cols":["modularity_Q"]}

    filled = audit_df.copy()
    for c in use_cols:
        filled[c] = filled[c].fillna(-1e9)

    mask = pareto_front(filled, use_cols)
    pf = audit_df[mask].copy().sort_values("beta")

    maxQ = pf["modularity_Q"].max()
    cand = pf[pf["modularity_Q"] >= (maxQ - 0.02)]
    if len(cand) == 0:
        chosen = pf.iloc[0]
        rule = "pareto_minbeta"
    else:
        chosen = cand.sort_values("beta").iloc[0]
        rule = "pareto_minbeta_with_Q_tolerance"

    return float(chosen["beta"]), {
        "rule": rule,
        "used_cols": use_cols,
        "pareto_betas": list(pf["beta"].astype(float).values),
        "maxQ": float(maxQ),
        "Q_tolerance": 0.02
    }

# ==========================================
# 0) Paths — usa cfg.run_root se existir, senão acha a run mais recente
# ==========================================
try:
    run_root = cfg.run_root
except Exception:
    base_dir = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"
    run_root = find_latest_run(os.path.join(base_dir, "runs"))
    if run_root is None:
        raise FileNotFoundError("Não encontrei runs em /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs")

part2_dir  = ensure_dir(os.path.join(run_root, "part2"))
inter_dir  = ensure_dir(os.path.join(run_root, "intermediate"))

nodes_path = os.path.join(inter_dir, "nodes_clean.csv")
edges_path = os.path.join(inter_dir, "edges_clean.csv")
if not (os.path.exists(nodes_path) and os.path.exists(edges_path)):
    raise FileNotFoundError("Não encontrei nodes_clean.csv/edges_clean.csv em intermediate. Rode a Parte 1 antes.")

nodes = pd.read_csv(nodes_path)
edges = pd.read_csv(edges_path)

if "nome_tokensort_norm" not in nodes.columns:
    raise KeyError("nodes_clean.csv precisa conter 'nome_tokensort_norm'.")
for c in ["u","v"]:
    if c not in edges.columns:
        raise KeyError("edges_clean.csv precisa conter colunas 'u' e 'v'.")
if "w" not in edges.columns:
    edges["w"] = 1.0

# ==========================================
# 1) Build FULL graph
# ==========================================
G_full = nx.Graph()
G_full.add_nodes_from(nodes["nome_tokensort_norm"].astype(str).tolist())

for _, r in edges.iterrows():
    u = str(r["u"]); v = str(r["v"])
    if not u or not v or u == v:
        continue
    w = float(r["w"]) if pd.notna(r["w"]) else 1.0
    if G_full.has_edge(u, v):
        G_full[u][v]["weight"] += w
    else:
        G_full.add_edge(u, v, weight=w)

components = sorted([len(c) for c in nx.connected_components(G_full)], reverse=True)
giant_size = components[0] if components else 0
summary_full = {
    "n_nodes_full": int(G_full.number_of_nodes()),
    "n_edges_full": int(G_full.number_of_edges()),
    "n_components_full": int(len(components)),
    "component_sizes_full_top10": components[:10],
    "giant_component_size_full": int(giant_size),
}
save_json(summary_full, os.path.join(part2_dir, "graphs_summary.json"))
print("FULL graph:", summary_full)

giant_nodes = max(nx.connected_components(G_full), key=len) if G_full.number_of_nodes() else set()
G_giant = G_full.subgraph(giant_nodes).copy()

summary_giant = {
    "n_nodes_giant": int(G_giant.number_of_nodes()),
    "n_edges_giant": int(G_giant.number_of_edges()),
}
save_json(summary_giant, os.path.join(part2_dir, "giant_summary.json"))
print("GIANT graph:", summary_giant)

# ==========================================
# 2) β sweep — Disparity Filter + Comunidades + Métricas + Coerência sociotécnica
# ==========================================
BETAS = [0.25, 0.50, 0.75, 1.00]
SEED  = 42

meta_cols = detect_sociotech_cols(nodes)
nodes_giant_df = nodes[nodes["nome_tokensort_norm"].isin(G_giant.nodes())].copy()
print("Sociotech cols detected:", meta_cols)

audit_rows = []
artifacts = {}

for beta in BETAS:
    t0 = time.time()

    H = disparity_filter_backbone(G_giant, beta=beta, weight="weight", or_rule=True)

    labels_map, algo = best_partition(H, weight="weight", seed=SEED, prefer="leiden")
    comms = communities_from_labels(labels_map)

    sizes = np.array([len(c) for c in comms], dtype=float)
    if sizes.sum() > 0:
        p = sizes / sizes.sum()
        H_entropy = -np.sum(p * np.log(p + 1e-15))
        H_norm = float(H_entropy / np.log(len(p) + 1e-15)) if len(p) > 1 else 0.0
    else:
        H_norm = 0.0

    m = graph_metrics(H, comms, weight="weight")

    df_lab = pd.DataFrame({
        "node": list(H.nodes()),
        "cluster": [labels_map[n] for n in H.nodes()]
    }).merge(nodes_giant_df, left_on="node", right_on="nome_tokensort_norm", how="left")

    V_UF = V_IES = np.nan
    p_UF = p_IES = np.nan

    if meta_cols.get("UF") and meta_cols["UF"] in df_lab.columns:
        V_UF, p_UF, _ = cramers_v(df_lab["cluster"], df_lab[meta_cols["UF"]].fillna("NA"))
    if meta_cols.get("IES") and meta_cols["IES"] in df_lab.columns:
        V_IES, p_IES, _ = cramers_v(df_lab["cluster"], df_lab[meta_cols["IES"]].fillna("NA"))

    row = {
        "beta": float(beta),
        "algo": algo,
        "n_nodes": int(H.number_of_nodes()),
        "n_edges": int(H.number_of_edges()),
        "n_components": int(nx.number_connected_components(H)),
        "giant_size": int(max([len(c) for c in nx.connected_components(H)], default=0)),
        "K": int(len(comms)),
        "H_norm": float(H_norm),
        "modularity_Q": m["modularity_Q"],
        "coverage": m["coverage"],
        "conductance_mean": m["conductance_mean"],
        "conductance_weighted_mean": m["conductance_weighted_mean"],
        "asymptotic_surprise": m["asymptotic_surprise"],
        "V_UF": float(V_UF) if V_UF==V_UF else np.nan,
        "p_UF": float(p_UF) if p_UF==p_UF else np.nan,
        "V_IES": float(V_IES) if V_IES==V_IES else np.nan,
        "p_IES": float(p_IES) if p_IES==p_IES else np.nan,
        "seconds": float(time.time() - t0),
    }
    audit_rows.append(row)

    artifacts[str(beta)] = {"backbone_edges": H.number_of_edges(), "K": len(comms), "algo": algo}

audit_df = pd.DataFrame(audit_rows).sort_values("beta")
audit_path = os.path.join(part2_dir, "backbone_audit.csv")
audit_df.to_csv(audit_path, index=False)

save_json({"artifacts": artifacts, "meta_cols": meta_cols}, os.path.join(part2_dir, "backbone_artifacts_summary.json"))

print("\n✅ backbone_audit.csv salvo em:", audit_path)
display(audit_df)

# ==========================================
# 3) Seleciona β* (Pareto + parcimônia)
# ==========================================
beta_star, beta_rule = select_beta_star(audit_df)
save_json({"beta_star": beta_star, "selection": beta_rule}, os.path.join(part2_dir, "beta_selection.json"))
print("\n✅ β* selecionado:", beta_star)
print("Regra:", beta_rule)

# ==========================================
# 4) Re-executa no β* e salva edges_beta + nodes_plus_clusters
# ==========================================
H_star = disparity_filter_backbone(G_giant, beta=beta_star, weight="weight", or_rule=True)
labels_star, algo_star = best_partition(H_star, weight="weight", seed=SEED, prefer="leiden")
comms_star = communities_from_labels(labels_star)
m_star = graph_metrics(H_star, comms_star, weight="weight")

edges_beta = [{"u": u, "v": v, "w": float(d.get("weight", 1.0))} for u, v, d in H_star.edges(data=True)]
edges_beta_df = pd.DataFrame(edges_beta)
edges_beta_out = os.path.join(part2_dir, "edges_beta.csv")
edges_beta_df.to_csv(edges_beta_out, index=False)

nodes_plus = nodes_giant_df.copy()
nodes_plus["cluster_mod"] = nodes_plus["nome_tokensort_norm"].map(lambda x: int(labels_star.get(x, -1)))
nodes_plus_out = os.path.join(part2_dir, "nodes_plus_clusters.csv")
nodes_plus.to_csv(nodes_plus_out, index=False)

save_json({
    "beta_star": beta_star,
    "algo": algo_star,
    "n_nodes": int(H_star.number_of_nodes()),
    "n_edges": int(H_star.number_of_edges()),
    "K": int(len(comms_star)),
    **{k: v for k,v in m_star.items() if k != "conductance_df"}
}, os.path.join(part2_dir, "community_metrics_beta_star.json"))

m_star["conductance_df"].to_csv(os.path.join(part2_dir, "conductance_by_cluster_beta_star.csv"), index=False)

print("\n✅ Salvos para reuso (Partes 3+):")
print(" -", edges_beta_out)
print(" -", nodes_plus_out)
print(" -", os.path.join(part2_dir, "community_metrics_beta_star.json"))
print(" -", os.path.join(part2_dir, "conductance_by_cluster_beta_star.csv"))

print("\n=== Resumo β* ===")
print("beta* =", beta_star, "| algo =", algo_star)
print("Nodes =", H_star.number_of_nodes(), "| Edges =", H_star.number_of_edges(), "| K =", len(comms_star))
print("Q =", round(m_star["modularity_Q"], 3),
      "| Coverage =", round(m_star["coverage"], 3),
      "| Conductance(mean) =", round(m_star["conductance_mean"], 3),
      "| Surprise =", round(m_star["asymptotic_surprise"], 1))

FULL graph: {'n_nodes_full': 510, 'n_edges_full': 1609, 'n_components_full': 99, 'component_sizes_full_top10': [404, 4, 2, 2, 2, 2, 2, 1, 1, 1], 'giant_component_size_full': 404}
GIANT graph: {'n_nodes_giant': 404, 'n_edges_giant': 1600}
Sociotech cols detected: {'UF': None, 'IES': 'SG_ENTIDADE_ENSINO', 'PROGRAMA': None, 'NOTA': None}

✅ backbone_audit.csv salvo em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part2/backbone_audit.csv


,beta,algo,n_nodes,n_edges,n_components,giant_size,K,H_norm,modularity_Q,coverage,conductance_mean,conductance_weighted_mean,asymptotic_surprise,V_UF,p_UF,V_IES,p_IES,seconds
0,0.25,louvain,404,433,82,272,96,0.820118,0.841796,0.929425,0.011575,0.070575,1087.761215,NaN,NaN,0.666476,1.316955e-122,1.022294
1,0.50,louvain,404,987,12,383,26,0.838802,0.752474,0.847692,0.088945,0.152308,1311.238608,NaN,NaN,0.480931,5.333772e-180,0.228234
2,0.75,louvain,404,1490,1,404,11,0.958399,0.708143,0.809332,0.188377,0.190668,1254.456376,NaN,NaN,0.654375,6.603409e-228,0.282726
3,1.00,louvain,404,1600,1,404,12,0.930700,0.701762,0.803230,0.200051,0.196770,1216.649232,NaN,NaN,0.636465,3.922156e-231,0.370903



✅ β* selecionado: 0.25
Regra: {'rule': 'pareto_minbeta_with_Q_tolerance', 'used_cols': ['modularity_Q', 'V_IES', 'H_norm', 'coverage'], 'pareto_betas': [np.float64(0.25), np.float64(0.5), np.float64(0.75)], 'maxQ': 0.8417959450133135, 'Q_tolerance': 0.02}

✅ Salvos para reuso (Partes 3+):
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part2/edges_beta.csv
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part2/nodes_plus_clusters.csv
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part2/community_metrics_beta_star.json
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part2/conductance_by_cluster_beta_star.csv

=== Resumo β* ===
beta* = 0.25 | algo = louvain
Nodes = 404 | Edges = 433 | K = 96
Q = 0.842 | Coverage = 0.929 | Conductance(mean) = 0.012 | Surprise = 1087.8


In [ ]:
# ============================================================
# PARTE 3 — CONSENSO / ESCOLHA + ESTABILIDADE
# - Corrige o problema prático do β*=0.25 (K=96 microclusters)
# - Seleciona uma configuração mais estudável
# - Gera: (i) auditoria comparativa, (ii) partição final,
#         (iii) estabilidade bootstrap (NMI/ARI), (iv) persistência par-a-par
# - Persiste tudo no Drive para reuso
# ============================================================

import os, re, json, math, time, itertools
import numpy as np
import pandas as pd
import networkx as nx

# ---------------------------
# 0) Utils
# ---------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def find_latest_run(root_base):
    if not os.path.exists(root_base):
        return None
    runs = [d for d in os.listdir(root_base) if d.startswith("run_")]
    if not runs:
        return None
    runs = sorted(runs)
    return os.path.join(root_base, runs[-1])

def pick_first_present(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def detect_sociotech_cols(nodes_df):
    cols = list(nodes_df.columns)
    cand = {
        "UF": ["SG_UF_PROGRAMA","uf","SG_UF","estado","state"],
        "IES": ["SG_ENTIDADE_ENSINO","NM_ENTIDADE_ENSINO","ies","instituicao","instituicao_original","institution"],
        "PROGRAMA": ["NM_PROGRAMA_IES","programa","nm_programa","program"],
        "NOTA": ["NOTA_PROGRAMA_IES","nota_programa","nota","capes_grade","grade"],
    }
    out = {}
    for k, cands in cand.items():
        out[k] = pick_first_present(cols, cands)
    return out

def cramers_v(x, y):
    # retorna (V, p)
    from scipy.stats import chi2_contingency
    tbl = pd.crosstab(pd.Series(x, dtype="category"), pd.Series(y, dtype="category"))
    if tbl.size == 0:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(tbl, correction=False)
    n = tbl.to_numpy().sum()
    r, c = tbl.shape
    denom = n * (min(r, c) - 1)
    if denom <= 0:
        return np.nan, float(p)
    V = math.sqrt(chi2 / denom)
    return float(V), float(p)

def communities_from_labels(labels_map):
    from collections import defaultdict
    groups = defaultdict(list)
    for n, c in labels_map.items():
        groups[int(c)].append(n)
    return [set(groups[k]) for k in sorted(groups.keys())]

# ---------------------------
# 1) Disparity Filter + métricas
# ---------------------------
def disparity_filter_backbone(G, beta=0.5, weight="weight", or_rule=True):
    """
    Undirected weighted disparity-filter backbone.
    alpha_ij = (1 - p_ij)^(k_i - 1)   for k_i>1
    p_ij = w_ij / s_i
    Keep if alpha_ij < beta (OR rule by endpoint).
    """
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))

    degree = {n: G.degree(n) for n in G.nodes()}
    strength = {}
    for n in G.nodes():
        s = 0.0
        for _, _, d in G.edges(n, data=True):
            s += float(d.get(weight, 1.0))
        strength[n] = s

    def alpha(i, w_ij):
        k = degree[i]
        if k <= 1:
            return 0.0
        s = strength[i]
        if s <= 0:
            return 1.0
        p = max(min(w_ij / s, 1.0), 0.0)
        return (1.0 - p) ** (k - 1)

    for u, v, d in G.edges(data=True):
        w = float(d.get(weight, 1.0))
        a_uv = alpha(u, w)
        a_vu = alpha(v, w)
        keep = (a_uv < beta) or (a_vu < beta) if or_rule else ((a_uv < beta) and (a_vu < beta))
        if keep:
            H.add_edge(u, v, **{weight: w})
    return H

def graph_metrics(G, communities, weight="weight"):
    from networkx.algorithms.community.quality import modularity

    Q = modularity(G, communities, weight=weight) if G.number_of_edges() > 0 else float("nan")

    total_w = sum(float(d.get(weight, 1.0)) for _,_,d in G.edges(data=True))
    if total_w <= 0:
        coverage = float("nan")
    else:
        w_in = 0.0
        for S in communities:
            SG = G.subgraph(S)
            w_in += sum(float(d.get(weight, 1.0)) for _,_,d in SG.edges(data=True))
        coverage = w_in / total_w

    strength = {n: 0.0 for n in G.nodes()}
    for u, v, d in G.edges(data=True):
        w = float(d.get(weight, 1.0))
        strength[u] += w
        strength[v] += w
    total_vol = sum(strength.values())

    def cut_weight(S):
        S = set(S)
        cw = 0.0
        for u, v, d in G.edges(data=True):
            if (u in S) ^ (v in S):
                cw += float(d.get(weight, 1.0))
        return cw

    rows = []
    for cid, S in enumerate(communities):
        S = set(S)
        vol = sum(strength[n] for n in S)
        cw = cut_weight(S)
        denom = min(vol, total_vol - vol)
        phi = (cw / denom) if denom and denom > 0 else 0.0
        rows.append([cid, len(S), vol, cw, phi])
    cond_df = pd.DataFrame(rows, columns=["community","size","volume","cut","conductance"])
    cond_mean = float(cond_df["conductance"].mean()) if len(cond_df) else float("nan")
    cond_wmean = float(np.average(cond_df["conductance"], weights=np.clip(cond_df["volume"].values, 1e-12, None))) if len(cond_df) else float("nan")

    # Asymptotic Surprise (aprox. não-ponderada)
    n = G.number_of_nodes()
    m = G.number_of_edges()
    if n < 2 or m < 1:
        S_asym = float("nan")
    else:
        M = n*(n-1)//2
        k_in = 0
        M_in = 0
        for S in communities:
            S = set(S)
            k_in += G.subgraph(S).number_of_edges()
            s = len(S)
            M_in += s*(s-1)//2
        eps = 1e-15
        p_in  = max(min(k_in / max(m, 1), 1-eps), eps)
        p_exp = max(min(M_in / max(M, 1), 1-eps), eps)
        S_asym = m * ( p_in*math.log(p_in/p_exp) + (1-p_in)*math.log((1-p_in)/(1-p_exp)) )

    return {
        "modularity_Q": float(Q),
        "coverage": float(coverage),
        "conductance_mean": float(cond_mean),
        "conductance_weighted_mean": float(cond_wmean),
        "asymptotic_surprise": float(S_asym),
        "conductance_df": cond_df
    }

# ---------------------------
# 2) Import robusto do python-louvain (evita shadowing)
# ---------------------------
def import_python_louvain():
    try:
        import community.community_louvain as community_louvain
        if not hasattr(community_louvain, "best_partition"):
            raise ImportError("community.community_louvain sem best_partition.")
        return community_louvain
    except Exception:
        import sys, subprocess, importlib
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "python-louvain"])
        community_louvain = importlib.import_module("community.community_louvain")
        if not hasattr(community_louvain, "best_partition"):
            raise ImportError("Falha ao importar python-louvain (best_partition ausente).")
        return community_louvain

community_louvain = import_python_louvain()

# ---------------------------
# 3) Louvain com resolução (para controlar K)
# ---------------------------
def louvain_partition(G, weight="weight", seed=42, resolution=1.0):
    if G.number_of_edges() == 0:
        return {u: i for i, u in enumerate(G.nodes())}
    labels = community_louvain.best_partition(
        G,
        weight=weight,
        random_state=seed,
        resolution=resolution
    )
    return {k: int(v) for k,v in labels.items()}

def entropy_norm_from_labels(labels_map):
    # normaliza entropia pelo log(K)
    labs = list(labels_map.values())
    if not labs:
        return 0.0
    counts = pd.Series(labs).value_counts().values.astype(float)
    p = counts / counts.sum()
    H = -np.sum(p * np.log(p + 1e-15))
    K = len(counts)
    return float(H / np.log(K + 1e-15)) if K > 1 else 0.0

def microcluster_stats(labels_map, min_size=5):
    s = pd.Series(labels_map).value_counts()
    n_micro = int((s < min_size).sum())
    share_micro = float(s[s < min_size].sum() / s.sum()) if s.sum() else 0.0
    return n_micro, share_micro

# ---------------------------
# 4) Bootstrap estabilidade + persistência
# ---------------------------
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

def bootstrap_graph_from_edges(G, sample_frac=1.0, seed=42):
    rng = np.random.default_rng(seed)
    edges = list(G.edges(data=True))
    m = len(edges)
    k = max(1, int(round(sample_frac * m)))
    idxs = rng.integers(0, m, size=k)  # com reposição
    sampled = [edges[i] for i in idxs]

    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    for u, v, d in sampled:
        w = float(d.get("weight", 1.0))
        if u == v:
            continue
        if H.has_edge(u, v):
            H[u][v]["weight"] += w
        else:
            H.add_edge(u, v, weight=w)
    return H

def nmi_ari(ref_labels, test_labels, nodes_list):
    y_ref = np.array([ref_labels[u] for u in nodes_list], dtype=int)
    y_hat = np.array([test_labels.get(u, -1) for u in nodes_list], dtype=int)
    return float(normalized_mutual_info_score(y_ref, y_hat)), float(adjusted_rand_score(y_ref, y_hat))

def pairwise_persistence(ref_labels, boot_labels_list, nodes_list, min_size=2):
    # persistência par-a-par apenas para pares no MESMO cluster na partição de referência
    idx = {u:i for i,u in enumerate(nodes_list)}
    # pares de referência por cluster
    ref_series = pd.Series(ref_labels)
    byc = {}
    for c, members in ref_series.groupby(ref_series).groups.items():
        mem = [idx[u] for u in members if u in idx]
        if len(mem) >= min_size:
            byc[int(c)] = sorted(mem)

    # conjunto global de pares de referência
    ref_pairs = set()
    comm_pairs = {}
    for c, mem in byc.items():
        pairs = set()
        for i in range(len(mem)):
            for j in range(i+1, len(mem)):
                p = (mem[i], mem[j])
                pairs.add(p)
                ref_pairs.add(p)
        comm_pairs[c] = pairs

    B = len(boot_labels_list)
    if B == 0 or len(ref_pairs) == 0:
        return {"global_mean": np.nan, "by_community": {}}

    counts = {p: 0 for p in ref_pairs}

    for lab in boot_labels_list:
        # agrupa por cluster no bootstrap
        inv = {}
        for u, c in lab.items():
            if u in idx:
                inv.setdefault(int(c), []).append(idx[u])
        for _, mem in inv.items():
            if len(mem) < 2:
                continue
            mem = sorted(mem)
            for i in range(len(mem)):
                for j in range(i+1, len(mem)):
                    p = (mem[i], mem[j])
                    if p in counts:
                        counts[p] += 1

    pers_vals = np.array([counts[p]/B for p in ref_pairs], dtype=float)
    global_mean = float(pers_vals.mean()) if pers_vals.size else np.nan

    by_comm = {}
    for c, pairs in comm_pairs.items():
        if not pairs:
            by_comm[c] = {"n_pairs": 0, "mean_persistence": np.nan}
            continue
        vals = np.array([counts[p]/B for p in pairs], dtype=float)
        by_comm[c] = {"n_pairs": int(len(pairs)), "mean_persistence": float(vals.mean())}
    return {"global_mean": global_mean, "by_community": by_comm}

# ============================================================
# A) Paths / Carrega dados gerados na Parte 1/2
# ============================================================
base_dir = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"
run_root = None
try:
    run_root = cfg.run_root
except Exception:
    run_root = find_latest_run(os.path.join(base_dir, "runs"))
if run_root is None:
    raise FileNotFoundError("Não encontrei run_root. Rode a Parte 1 e 2 antes.")

part2_dir = os.path.join(run_root, "part2")
part3_dir = ensure_dir(os.path.join(run_root, "part3"))
inter_dir = os.path.join(run_root, "intermediate")

nodes_clean_path = os.path.join(inter_dir, "nodes_clean.csv")
edges_clean_path = os.path.join(inter_dir, "edges_clean.csv")
if not (os.path.exists(nodes_clean_path) and os.path.exists(edges_clean_path)):
    raise FileNotFoundError("Não encontrei nodes_clean.csv / edges_clean.csv em intermediate (Parte 1).")

nodes = pd.read_csv(nodes_clean_path)
edges = pd.read_csv(edges_clean_path)

if "nome_tokensort_norm" not in nodes.columns:
    raise KeyError("nodes_clean.csv precisa conter 'nome_tokensort_norm'.")
if not all(c in edges.columns for c in ["u","v"]):
    raise KeyError("edges_clean.csv precisa conter colunas 'u' e 'v'.")
if "w" not in edges.columns:
    edges["w"] = 1.0

meta_cols = detect_sociotech_cols(nodes)
print("Sociotech cols (detectadas):", meta_cols)

# ============================================================
# B) Reconstrói FULL e GIANT (para Part 3 ser reprodutível)
# ============================================================
G_full = nx.Graph()
G_full.add_nodes_from(nodes["nome_tokensort_norm"].astype(str).tolist())
for _, r in edges.iterrows():
    u = str(r["u"]); v = str(r["v"])
    if not u or not v or u == v:
        continue
    w = float(r["w"]) if pd.notna(r["w"]) else 1.0
    if G_full.has_edge(u, v):
        G_full[u][v]["weight"] += w
    else:
        G_full.add_edge(u, v, weight=w)

components = sorted([len(c) for c in nx.connected_components(G_full)], reverse=True)
giant_nodes = max(nx.connected_components(G_full), key=len) if components else set()
G_giant = G_full.subgraph(giant_nodes).copy()

print("FULL:", {"n": G_full.number_of_nodes(), "m": G_full.number_of_edges(), "components": len(components), "giant": len(giant_nodes)})
print("GIANT:", {"n": G_giant.number_of_nodes(), "m": G_giant.number_of_edges()})

nodes_giant_df = nodes[nodes["nome_tokensort_norm"].isin(G_giant.nodes())].copy()

# ============================================================
# C) Seleção "paper-ready": controla K, microclusters e coerência IES
# (isso é o ajuste central que seu resultado mostrou ser necessário)
# ============================================================
BETAS = [0.25, 0.50, 0.75, 1.00]      # regimes de backbone
RESOLUTIONS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]  # controla granularidade do Louvain
TARGET_K = 16
MIN_CLUSTER_SIZE = 5
SEED = 42

audit_rows = []

for beta in BETAS:
    H = disparity_filter_backbone(G_giant, beta=beta, weight="weight", or_rule=True)
    for res in RESOLUTIONS:
        labels = louvain_partition(H, weight="weight", seed=SEED, resolution=res)
        comms = communities_from_labels(labels)
        K = len(comms)

        m = graph_metrics(H, comms, weight="weight")

        # entropia de tamanhos (evita degenerar em 1 cluster)
        Hn = entropy_norm_from_labels(labels)

        # microclusters (penaliza fragmentação)
        n_micro, share_micro = microcluster_stats(labels, min_size=MIN_CLUSTER_SIZE)

        # coerência IES (se existir)
        V_IES = np.nan
        p_IES = np.nan
        if meta_cols.get("IES") and meta_cols["IES"] in nodes_giant_df.columns:
            df_lab = pd.DataFrame({"node": list(H.nodes()), "cluster": [labels[n] for n in H.nodes()]}) \
                        .merge(nodes_giant_df, left_on="node", right_on="nome_tokensort_norm", how="left")
            V_IES, p_IES = cramers_v(df_lab["cluster"], df_lab[meta_cols["IES"]].fillna("NA"))

        # score composto (interpretable, "article-first"):
        # - favorece modularidade + coerência IES + coverage
        # - penaliza: distância de TARGET_K + share_micro
        k_pen = abs(K - TARGET_K) / max(TARGET_K, 1)
        score = (
            1.00 * m["modularity_Q"]
            + 0.35 * (0.0 if np.isnan(V_IES) else V_IES)
            + 0.20 * m["coverage"]
            + 0.10 * Hn
            - 0.50 * k_pen
            - 0.80 * share_micro
        )

        audit_rows.append({
            "beta": float(beta),
            "resolution": float(res),
            "n_nodes": int(H.number_of_nodes()),
            "n_edges": int(H.number_of_edges()),
            "K": int(K),
            "micro_clusters(<5)": int(n_micro),
            "micro_share": float(share_micro),
            "H_norm": float(Hn),
            "Q": float(m["modularity_Q"]),
            "coverage": float(m["coverage"]),
            "cond_mean": float(m["conductance_mean"]),
            "surprise": float(m["asymptotic_surprise"]),
            "V_IES": float(V_IES) if V_IES==V_IES else np.nan,
            "p_IES": float(p_IES) if p_IES==p_IES else np.nan,
            "score": float(score),
        })

audit_df = pd.DataFrame(audit_rows).sort_values("score", ascending=False)
audit_out = os.path.join(part3_dir, "paper_ready_selection_audit.csv")
audit_df.to_csv(audit_out, index=False)
print("\n✅ Audit salvo:", audit_out)
display(audit_df.head(20))

best = audit_df.iloc[0].to_dict()
print("\n✅ Melhor configuração (paper-ready):")
print(best)

beta_star = float(best["beta"])
res_star  = float(best["resolution"])

# ============================================================
# D) Aplica configuração escolhida e exporta artefatos "v3"
# ============================================================
H_star = disparity_filter_backbone(G_giant, beta=beta_star, weight="weight", or_rule=True)
labels_star = louvain_partition(H_star, weight="weight", seed=SEED, resolution=res_star)
comms_star = communities_from_labels(labels_star)
m_star = graph_metrics(H_star, comms_star, weight="weight")

# tabela de tamanhos
sizes = pd.Series(labels_star).value_counts().sort_index()
sizes_df = sizes.reset_index()
sizes_df.columns = ["cluster_mod","size"]
sizes_df = sizes_df.sort_values("size", ascending=False)
sizes_df.to_csv(os.path.join(part3_dir, "cluster_sizes_v3.csv"), index=False)

# condutância por cluster (paper-friendly)
cond_df = m_star["conductance_df"].copy()
cond_df = cond_df.rename(columns={"community":"cluster_mod"})
cond_df.to_csv(os.path.join(part3_dir, "conductance_by_cluster_v3.csv"), index=False)

# edges_beta_v3
edges_beta = [{"u": u, "v": v, "w": float(d.get("weight", 1.0))} for u, v, d in H_star.edges(data=True)]
edges_beta_df = pd.DataFrame(edges_beta)
edges_beta_out = os.path.join(part3_dir, "edges_beta_v3.csv")
edges_beta_df.to_csv(edges_beta_out, index=False)

# nodes_plus_clusters_v3
nodes_plus = nodes_giant_df.copy()
nodes_plus["cluster_mod"] = nodes_plus["nome_tokensort_norm"].map(lambda x: int(labels_star.get(x, -1)))
nodes_plus_out = os.path.join(part3_dir, "nodes_plus_clusters_v3.csv")
nodes_plus.to_csv(nodes_plus_out, index=False)

overall = {
    "selected_beta": beta_star,
    "selected_resolution": res_star,
    "n_nodes": int(H_star.number_of_nodes()),
    "n_edges": int(H_star.number_of_edges()),
    "K": int(len(comms_star)),
    "Q": float(m_star["modularity_Q"]),
    "coverage": float(m_star["coverage"]),
    "conductance_mean": float(m_star["conductance_mean"]),
    "conductance_weighted_mean": float(m_star["conductance_weighted_mean"]),
    "asymptotic_surprise": float(m_star["asymptotic_surprise"]),
}
save_json(overall, os.path.join(part3_dir, "community_metrics_overall_v3.json"))

print("\n✅ Artefatos v3 salvos (para Partes 4+):")
print(" -", nodes_plus_out)
print(" -", edges_beta_out)
print(" -", os.path.join(part3_dir, "community_metrics_overall_v3.json"))
print(" -", os.path.join(part3_dir, "conductance_by_cluster_v3.csv"))
print(" -", os.path.join(part3_dir, "cluster_sizes_v3.csv"))
print("\nResumo v3:", overall)

# ============================================================
# E) Estabilidade: edge bootstrap + NMI/ARI + persistência par-a-par
# ============================================================
B = 50
SAMPLE_FRAC = 1.0   # com reposição; 1.0 ~ mesmo nº de arestas em média
SEED0 = 42

nodes_list = list(H_star.nodes())
ref_labels = {u: int(labels_star[u]) for u in nodes_list}

nmi_list = []
ari_list = []
boot_labels_list = []

for b in range(B):
    Hb = bootstrap_graph_from_edges(H_star, sample_frac=SAMPLE_FRAC, seed=SEED0 + b)
    # usa MESMA resolução para consistência
    lab_b = louvain_partition(Hb, weight="weight", seed=SEED0 + b, resolution=res_star)
    boot_labels_list.append(lab_b)

    nmi_b, ari_b = nmi_ari(ref_labels, lab_b, nodes_list)
    nmi_list.append(nmi_b)
    ari_list.append(ari_b)

stab_df = pd.DataFrame({
    "bootstrap": np.arange(B, dtype=int),
    "nmi_to_reference": nmi_list,
    "ari_to_reference": ari_list
})
stab_out = os.path.join(part3_dir, "bootstrap_stability_v3.csv")
stab_df.to_csv(stab_out, index=False)

persistence = pairwise_persistence(ref_labels, boot_labels_list, nodes_list)
save_json(persistence, os.path.join(part3_dir, "bootstrap_persistence_v3.json"))

# ============================================================
# F) Prints "paper-style"
# ============================================================
def show_table(title, df):
    print(title + "\n" + "="*len(title))
    print(df.to_string(index=False))
    print()

tblA = pd.DataFrame([
    ["Nodes", overall["n_nodes"]],
    ["Edges", overall["n_edges"]],
    ["Communities (K)", overall["K"]],
    ["Selected beta", f"{overall['selected_beta']:.2f}"],
    ["Selected resolution", f"{overall['selected_resolution']:.2f}"],
    ["Modularity (Q)", f"{overall['Q']:.3f}"],
    ["Coverage", f"{overall['coverage']:.3f}"],
    ["Conductance (mean)", f"{overall['conductance_mean']:.3f}"],
    ["Conductance (weighted mean)", f"{overall['conductance_weighted_mean']:.3f}"],
    ["Asymptotic Surprise (S)", f"{overall['asymptotic_surprise']:.1f}"],
], columns=["Metric","Value"])

show_table("Table A — Community quality (paper-ready selection)", tblA)

cond_show = cond_df.copy()
cond_show["conductance"] = cond_show["conductance"].map(lambda x: f"{x:.3f}")
cond_show["volume"] = cond_show["volume"].map(lambda x: f"{x:.1f}")
cond_show["cut"] = cond_show["cut"].map(lambda x: f"{x:.1f}")
cond_show = cond_show.rename(columns={"size":"Size","volume":"Volume","cut":"Cut","conductance":"Conductance"})
cond_show = cond_show.sort_values("Conductance")
show_table("Table B — Conductance by community (lower is better)", cond_show)

stab_tbl = pd.DataFrame({
    "Metric": ["NMI (mean ± sd)", "ARI (mean ± sd)"],
    "Value": [
        f"{np.mean(nmi_list):.3f} ± {np.std(nmi_list, ddof=1):.3f}",
        f"{np.mean(ari_list):.3f} ± {np.std(ari_list, ddof=1):.3f}",
    ]
})
show_table("Table C — Stability under edge bootstrap (Louvain vs reference)", stab_tbl)

rowsD = [["Global pairwise persistence", persistence["global_mean"]]]
for cid, info in persistence["by_community"].items():
    rowsD.append([f"Cluster {cid} — mean persistence (n_pairs={info['n_pairs']})", info["mean_persistence"]])
dfD = pd.DataFrame(rowsD, columns=["Statistic","Value"])
dfD["Value"] = dfD["Value"].map(lambda x: f"{x:.3f}" if isinstance(x,(float,int)) and x==x else "nan")
show_table("Table D — Pairwise co-assignment persistence", dfD)

print("✅ Arquivos salvos em:", part3_dir)
print(" -", audit_out)
print(" -", nodes_plus_out)
print(" -", edges_beta_out)
print(" -", stab_out)
print(" -", os.path.join(part3_dir, "bootstrap_persistence_v3.json"))


Sociotech cols (detectadas): {'UF': None, 'IES': 'SG_ENTIDADE_ENSINO', 'PROGRAMA': None, 'NOTA': None}
FULL: {'n': 510, 'm': 1609, 'components': 99, 'giant': 404}
GIANT: {'n': 404, 'm': 1600}

✅ Audit salvo: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/paper_ready_selection_audit.csv


,beta,resolution,n_nodes,n_edges,K,micro_clusters(<5),micro_share,H_norm,Q,coverage,cond_mean,surprise,V_IES,p_IES,score
19,0.75,1.5,404,1490,16,1,0.007426,0.962699,0.704005,0.783875,0.224558,1404.266644,0.621121,2.077422e-292,1.168502
18,0.75,1.2,404,1490,15,0,0.000000,0.979554,0.705828,0.791015,0.217406,1469.154981,0.631708,4.805466e-287,1.151834
25,1.00,1.2,404,1600,15,0,0.000000,0.967175,0.698348,0.784449,0.223560,1441.454201,0.644522,2.420404e-304,1.146288
26,1.00,1.5,404,1600,15,0,0.000000,0.963747,0.698553,0.780394,0.228734,1385.579089,0.635721,2.058845e-292,1.142258
16,0.75,0.8,404,1490,14,0,0.000000,0.967052,0.703802,0.800534,0.216141,1401.252554,0.633978,1.485563e-269,1.120006
15,0.75,0.6,404,1490,14,0,0.000000,0.949386,0.698514,0.807457,0.226269,1380.308839,0.629831,2.007974e-264,1.112885
23,1.00,0.8,404,1600,13,0,0.000000,0.966554,0.701432,0.799317,0.205246,1375.336453,0.656527,1.632384e-275,1.093985
22,1.00,0.6,404,1600,14,2,0.014851,0.896568,0.691564,0.815821,0.198588,1315.897947,0.611227,5.915141e-242,1.083933
24,1.00,1.0,404,1600,12,1,0.009901,0.930700,0.701762,0.803230,0.200051,1216.649232,0.636465,3.922156e-231,1.045320
17,0.75,1.0,404,1490,11,0,0.000000,0.958399,0.708143,0.809332,0.188377,1254.456376,0.654375,6.603409e-228,1.038631



✅ Melhor configuração (paper-ready):
{'beta': 0.75, 'resolution': 1.5, 'n_nodes': 404.0, 'n_edges': 1490.0, 'K': 16.0, 'micro_clusters(<5)': 1.0, 'micro_share': 0.007425742574257425, 'H_norm': 0.962699167233339, 'Q': 0.7040050180742021, 'coverage': 0.7838753876108747, 'cond_mean': 0.2245577230973673, 'surprise': 1404.266643929678, 'V_IES': 0.6211205883612462, 'p_IES': 2.0774224680341565e-292, 'score': 1.1685016241867412}

✅ Artefatos v3 salvos (para Partes 4+):
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/nodes_plus_clusters_v3.csv
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/edges_beta_v3.csv
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/community_metrics_overall_v3.json
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/conductance_by_cluster_v3.csv
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/cluster_sizes_

In [ ]:
# ============================================================
# PARTE 4 (V5) — Master table + associações + Kruskal (robusto)
# Corrige:
#  - não depende de autor_id em autores_final
#  - reconstrói works_2010 / citations_2010 / active_years_since2010
#    a partir de producoes_final.xlsx (coluna "autores" + "ano" + "citations")
#  - evita erro "All numbers are identical in kruskal"
#  - salva tudo no Drive para reuso
# ============================================================

import os, re, json, math
import numpy as np
import pandas as pd
import networkx as nx

from collections import defaultdict
from datetime import datetime

# -----------------------------
# 0) Utilitários
# -----------------------------
def normalize_name_token_sort(s: str) -> str:
    import unicodedata
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    s = re.sub(r"[^a-z\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 2]
    toks.sort()
    return " ".join(toks)

def most_recent_run(base_runs_dir):
    if not os.path.isdir(base_runs_dir):
        return None
    runs = [d for d in os.listdir(base_runs_dir) if d.startswith("run_") and os.path.isdir(os.path.join(base_runs_dir, d))]
    if not runs:
        return None
    runs_sorted = sorted(runs, key=lambda x: x, reverse=True)
    return os.path.join(base_runs_dir, runs_sorted[0])

def find_any(path_dir, candidates):
    """Encontra o primeiro arquivo existente em path_dir com um dos nomes candidatos."""
    for fn in candidates:
        p = os.path.join(path_dir, fn)
        if os.path.exists(p):
            return p
    return None

def find_uploaded_xlsx(input_dir, stem):
    """
    Achar arquivos como:
      autores_final.xlsx, autores_final (1).xlsx, autores_final (2).xlsx
    """
    if not os.path.isdir(input_dir):
        return None
    pats = [
        f"{stem}.xlsx",
        f"{stem} (1).xlsx",
        f"{stem} (2).xlsx",
        f"{stem} (3).xlsx",
    ]
    for fn in pats:
        p = os.path.join(input_dir, fn)
        if os.path.exists(p):
            return p
    # fallback: qualquer arquivo que comece com stem e termine .xlsx
    hits = [f for f in os.listdir(input_dir) if f.lower().startswith(stem.lower()) and f.lower().endswith(".xlsx")]
    if hits:
        hits = sorted(hits)
        return os.path.join(input_dir, hits[0])
    return None

def detect_col(df, candidates):
    """Retorna o primeiro candidato presente (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_authors_cell(x):
    """
    Parse robusto para a coluna 'autores' (producoes_final).
    Suporta:
      - string com ';'
      - string com ','
      - string tipo "['A', 'B']"
      - string tipo JSON
    Retorna lista de strings (nomes).
    """
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []

    # tenta JSON/list-like
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            import ast
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple)):
                return [str(t).strip() for t in obj if str(t).strip()]
            if isinstance(obj, dict):
                # se vier dict com nomes em alguma chave
                vals = []
                for v in obj.values():
                    if isinstance(v, (list, tuple)):
                        vals += [str(t).strip() for t in v]
                    else:
                        vals.append(str(v).strip())
                return [v for v in vals if v]
        except Exception:
            pass

    # split por delimitadores comuns
    if ";" in s:
        parts = [p.strip() for p in s.split(";")]
        return [p for p in parts if p]
    if " | " in s:
        parts = [p.strip() for p in s.split(" | ")]
        return [p for p in parts if p]
    if "," in s:
        # cuidado: nomes podem ter vírgula "SOBRENOME, Nome"
        # heurística: se tiver muitas vírgulas, assume lista; senão mantém como 1 nome
        if s.count(",") >= 2:
            parts = [p.strip() for p in s.split(",")]
            return [p for p in parts if p]
        else:
            return [s]

    return [s]

def cramers_v_with_pvalue(x, y):
    """
    Cramér's V + p-valor (chi2).
    Retorna (V, p).
    """
    from scipy.stats import chi2_contingency
    ct = pd.crosstab(x, y)
    if ct.size == 0 or ct.shape[0] < 2 or ct.shape[1] < 2:
        return (np.nan, np.nan)
    chi2, p, dof, exp = chi2_contingency(ct, correction=False)
    n = ct.to_numpy().sum()
    r, k = ct.shape
    denom = n * (min(r, k) - 1)
    if denom <= 0:
        return (np.nan, p)
    V = math.sqrt(chi2 / denom)
    return (V, p)

def kruskal_by_cluster_robust(df, cluster_col, value_col):
    """
    Kruskal-Wallis robusto:
      - se globalmente constante => retorna NA + note=constant_global
      - se <2 grupos com variância => retorna NA + note=insufficient_groups
      - captura erro "All numbers are identical in kruskal"
    """
    from scipy.stats import kruskal

    s = df[[cluster_col, value_col]].dropna()
    if s.empty:
        return {"value": value_col, "H": np.nan, "p": np.nan, "k_groups": 0, "note": "empty_after_dropna"}

    # global constant?
    if s[value_col].nunique(dropna=True) < 2:
        return {"value": value_col, "H": np.nan, "p": np.nan, "k_groups": 0, "note": "constant_global"}

    # agrupa por cluster
    groups = []
    for g, sub in s.groupby(cluster_col):
        vals = sub[value_col].values
        # precisa de pelo menos 2 valores e alguma variabilidade
        if len(vals) >= 2 and np.nanmin(vals) != np.nanmax(vals):
            groups.append(vals)

    if len(groups) < 2:
        return {"value": value_col, "H": np.nan, "p": np.nan, "k_groups": len(groups), "note": "insufficient_groups"}

    try:
        H, p = kruskal(*groups, nan_policy="omit")
        return {"value": value_col, "H": float(H), "p": float(p), "k_groups": int(len(groups)), "note": "ok"}
    except ValueError as e:
        # tipicamente: All numbers are identical in kruskal
        msg = str(e).strip().lower()
        if "identical" in msg:
            return {"value": value_col, "H": np.nan, "p": np.nan, "k_groups": int(len(groups)), "note": "identical_in_kruskal"}
        return {"value": value_col, "H": np.nan, "p": np.nan, "k_groups": int(len(groups)), "note": f"kruskal_error:{str(e)[:60]}"}

def fmt_p(p):
    if pd.isna(p):
        return "NA"
    if p == 0:
        return "0"
    if p < 1e-300:
        return "<1e-300"
    if p < 1e-3:
        return f"{p:.1e}"
    return f"{p:.4f}"

# -----------------------------
# 1) Caminhos (Drive / RUN)
# -----------------------------
DRIVE_ROOT = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"
RUNS_DIR = os.path.join(DRIVE_ROOT, "runs")

# Se você quiser forçar um run específico, defina aqui:
# RUN_DIR = os.path.join(RUNS_DIR, "run_20260202_175407")
RUN_DIR = most_recent_run(RUNS_DIR)
if RUN_DIR is None:
    raise FileNotFoundError(f"Não encontrei runs em: {RUNS_DIR}")

PART3_DIR = os.path.join(RUN_DIR, "part3")
PART4_DIR = os.path.join(RUN_DIR, "part4")
INPUT_DIR = os.path.join(RUN_DIR, "input")

os.makedirs(PART4_DIR, exist_ok=True)

# -----------------------------
# 2) Inputs v3 (paper-ready)
# -----------------------------
nodes_path = find_any(PART3_DIR, ["nodes_plus_clusters_v3.csv"])
edges_path = find_any(PART3_DIR, ["edges_beta_v3.csv"])

if nodes_path is None or edges_path is None:
    raise FileNotFoundError(
        "Não encontrei nodes_plus_clusters_v3.csv e/ou edges_beta_v3.csv em part3. "
        "Confirme se a PARTE 3 salvou os artefatos v3."
    )

nodes = pd.read_csv(nodes_path)
edges = pd.read_csv(edges_path)

# Colunas mínimas
if "nome_tokensort_norm" not in nodes.columns:
    # tenta derivar
    raw_name_col = detect_col(nodes, ["NM_DOCENTE", "nm_docente", "nome", "name", "nome_pesquisado"])
    if raw_name_col is None:
        raise KeyError("nodes_plus_clusters_v3.csv precisa ter 'nome_tokensort_norm' ou uma coluna de nome para derivar.")
    nodes["nome_tokensort_norm"] = nodes[raw_name_col].map(normalize_name_token_sort)

if "cluster_mod" not in nodes.columns:
    raise KeyError("nodes_plus_clusters_v3.csv precisa conter 'cluster_mod' (rótulo de comunidade).")

# Detecta IES no nodes (se existir)
ies_col_nodes = detect_col(nodes, ["SG_ENTIDADE_ENSINO", "IES", "instituicao", "instituicao_original", "NM_ENTIDADE_ENSINO"])
# (UF/PROGRAMA/NOTA podem não existir — OK)
uf_col_nodes = detect_col(nodes, ["SG_UF_PROGRAMA", "UF", "uf"])
prog_col_nodes = detect_col(nodes, ["NM_PROGRAMA_IES", "programa", "nm_programa_ies"])
nota_col_nodes = detect_col(nodes, ["NOTA_PROGRAMA_IES", "nota", "nota_programa"])

doc_set = set(nodes["nome_tokensort_norm"].astype(str))

print("PARTE 4 (V5) — inputs:")
print(f" - nodes: {nodes.shape} | edges: {edges.shape}")
print(f" - K: {nodes['cluster_mod'].nunique()} | N: {len(doc_set)}")
print("Sociotech cols (detectadas):", {"UF": uf_col_nodes, "IES": ies_col_nodes, "PROGRAMA": prog_col_nodes, "NOTA": nota_col_nodes})

# -----------------------------
# 3) Grafo + métricas de rede
# -----------------------------
# Detecta colunas de aresta
u_col = detect_col(edges, ["u", "source", "src", "from", "author_u", "node_u", "a"])
v_col = detect_col(edges, ["v", "target", "dst", "to", "author_v", "node_v", "b"])
w_col = detect_col(edges, ["w", "weight", "edge_weight", "count", "freq", "strength"])

if u_col is None or v_col is None:
    # fallback: assume as 2 primeiras colunas
    u_col, v_col = edges.columns[:2]
if w_col is None:
    w_col = None

E = pd.DataFrame({
    "u": edges[u_col].astype(str).map(normalize_name_token_sort),
    "v": edges[v_col].astype(str).map(normalize_name_token_sort),
})
E["w"] = edges[w_col] if w_col else 1.0
E = E[(E["u"].isin(doc_set)) & (E["v"].isin(doc_set)) & (E["u"] != E["v"])].copy()
E["w"] = pd.to_numeric(E["w"], errors="coerce").fillna(1.0)
E = E.groupby(["u", "v"], as_index=False)["w"].sum()

G = nx.Graph()
G.add_nodes_from(list(doc_set))
for _, r in E.iterrows():
    G.add_edge(r["u"], r["v"], weight=float(r["w"]))

# Degree/Strength
deg = dict(G.degree())
strength = {n: 0.0 for n in G.nodes()}
for a, b, d in G.edges(data=True):
    w = float(d.get("weight", 1.0))
    strength[a] += w
    strength[b] += w

net_df = pd.DataFrame({
    "nome_tokensort_norm": list(G.nodes()),
    "degree": [deg[n] for n in G.nodes()],
    "strength": [strength[n] for n in G.nodes()],
})
net_df["avg_tie_strength"] = net_df.apply(lambda r: (r["strength"] / r["degree"]) if r["degree"] > 0 else 0.0, axis=1)
net_df["log1p_degree"] = np.log1p(net_df["degree"].astype(float))
net_df["log1p_strength"] = np.log1p(net_df["strength"].astype(float))

# -----------------------------
# 4) Produtividade 2010+ reconstruída (SEM depender de autor_id)
# -----------------------------
# Lê producoes_final e computa works/citations/active_years por autor (nome normalizado)
prod_path = find_uploaded_xlsx(INPUT_DIR, "producoes_final")
if prod_path is None:
    raise FileNotFoundError(f"Não encontrei producoes_final*.xlsx em: {INPUT_DIR}")

producoes = pd.read_excel(prod_path)

col_ano = detect_col(producoes, ["ano", "year"])
col_autores = detect_col(producoes, ["autores", "authors"])
col_cit = detect_col(producoes, ["citations", "cited_by_count", "n_citations", "cits"])

if col_ano is None or col_autores is None:
    raise KeyError("producoes_final precisa ter colunas equivalentes a: 'ano' e 'autores'.")

# normaliza types
producoes[col_ano] = pd.to_numeric(producoes[col_ano], errors="coerce")
if col_cit is None:
    producoes["_cit"] = 0.0
    col_cit = "_cit"
else:
    producoes[col_cit] = pd.to_numeric(producoes[col_cit], errors="coerce").fillna(0.0)

# filtra 2010+
p2010 = producoes.loc[producoes[col_ano] >= 2010, [col_ano, col_autores, col_cit]].copy()
p2010 = p2010.dropna(subset=[col_ano, col_autores])

# acumula
works = defaultdict(int)
cits = defaultdict(float)
years_active = defaultdict(set)

for _, row in p2010.iterrows():
    y = int(row[col_ano])
    citv = float(row[col_cit]) if not pd.isna(row[col_cit]) else 0.0
    auth_list = parse_authors_cell(row[col_autores])
    # normaliza e filtra para PPGCI nodes
    auth_norm = [normalize_name_token_sort(a) for a in auth_list]
    auth_norm = [a for a in auth_norm if a in doc_set and a]  # somente nós do backbone
    if not auth_norm:
        continue
    # conta 1 work para cada autor do PPGCI que aparece na publicação
    for a in set(auth_norm):
        works[a] += 1
        cits[a] += citv
        years_active[a].add(y)

prod_df = pd.DataFrame({
    "nome_tokensort_norm": list(doc_set),
    "works_2010": [works.get(a, 0) for a in doc_set],
    "citations_2010": [cits.get(a, 0.0) for a in doc_set],
    "active_years_since2010": [len(years_active.get(a, set())) for a in doc_set],
})

# papers_count_ppgci_internal:
# aqui, como estamos no recorte intra-PPGCI, uma proxy consistente é:
# número de works (2010+) onde o autor aparece E há >=2 autores PPGCI no paper.
# Vamos recomputar isso por work:
papers_internal = defaultdict(int)

for _, row in p2010.iterrows():
    auth_list = parse_authors_cell(row[col_autores])
    auth_norm = [normalize_name_token_sort(a) for a in auth_list]
    auth_norm = [a for a in auth_norm if a in doc_set and a]
    if len(set(auth_norm)) >= 2:
        for a in set(auth_norm):
            papers_internal[a] += 1

prod_df["papers_count_ppgci_internal"] = [papers_internal.get(a, 0) for a in doc_set]

# métricas derivadas
prod_df["cits_per_work"] = prod_df.apply(lambda r: (r["citations_2010"] / r["works_2010"]) if r["works_2010"] > 0 else 0.0, axis=1)

print(f"✅ producoes_final: {os.path.basename(prod_path)} | shape: {producoes.shape}")
print(f"✅ rebuilt produtividade (2010+): {prod_df.shape} | via coluna autores (nome_tokensort_norm)")

# -----------------------------
# 5) Master table (merge)
# -----------------------------
base_cols = ["nome_tokensort_norm", "cluster_mod"]
keep_cols = base_cols[:]
for c in [ies_col_nodes, uf_col_nodes, prog_col_nodes, nota_col_nodes]:
    if c is not None and c in nodes.columns and c not in keep_cols:
        keep_cols.append(c)

nodes_small = nodes[keep_cols].copy()

master = (nodes_small
          .merge(net_df, on="nome_tokensort_norm", how="left")
          .merge(prod_df, on="nome_tokensort_norm", how="left"))

# preenche NAs
for c in ["degree","strength","avg_tie_strength","log1p_degree","log1p_strength",
          "works_2010","citations_2010","active_years_since2010","papers_count_ppgci_internal","cits_per_work"]:
    if c in master.columns:
        master[c] = master[c].fillna(0)

# garante tipos
master["cluster_mod"] = pd.to_numeric(master["cluster_mod"], errors="coerce").astype("Int64")

# salva master
master_path = os.path.join(PART4_DIR, "master_node_table_v3.csv")
master.to_csv(master_path, index=False)
print("✅ master_node_table_v3.csv salvo:", master_path)
print("Master shape:", master.shape)

# -----------------------------
# 6) Associação cluster ~ IES (Cramér's V)
# -----------------------------
assoc_rows = []
if ies_col_nodes is not None and ies_col_nodes in master.columns:
    V, p = cramers_v_with_pvalue(master["cluster_mod"], master[ies_col_nodes])
    assoc_rows.append({
        "x": "cluster_mod",
        "y": ies_col_nodes,
        "cramers_V": V,
        "p_value": p,
        "p_fmt": fmt_p(p)
    })
    print(f"✅ Associação (cluster ~ IES): V = {V:.3f} | p = {fmt_p(p)}")
else:
    print("⚠️ Sem coluna IES no nodes_plus_clusters_v3 — pulando Cramér's V (cluster~IES).")

assoc_df = pd.DataFrame(assoc_rows)
assoc_path = os.path.join(PART4_DIR, "cluster_associations_v3.csv")
assoc_df.to_csv(assoc_path, index=False)
print("✅ cluster_associations_v3.csv salvo:", assoc_path)

# Top IES por cluster (se IES existir)
if ies_col_nodes is not None and ies_col_nodes in master.columns:
    top_rows = []
    for k, sub in master.dropna(subset=[ies_col_nodes]).groupby("cluster_mod"):
        vc = sub[ies_col_nodes].value_counts().head(10)
        total = int(vc.sum())
        for ies, cnt in vc.items():
            top_rows.append({
                "cluster_mod": int(k),
                "IES": ies,
                "count": int(cnt),
                "share_in_cluster": float(cnt / max(sub.shape[0], 1)),
            })
    top_df = pd.DataFrame(top_rows)
    top_path = os.path.join(PART4_DIR, "top_ies_by_cluster_v3.csv")
    top_df.to_csv(top_path, index=False)
    print("✅ top_ies_by_cluster_v3.csv salvo:", top_path)

# -----------------------------
# 7) Kruskal-Wallis por cluster (ROBUST)
# -----------------------------
numeric_vars = [
    "works_2010",
    "citations_2010",
    "active_years_since2010",
    "papers_count_ppgci_internal",
    "cits_per_work",
    "strength",
    "degree",
    "avg_tie_strength",
]

kw_rows = []
for v in numeric_vars:
    if v in master.columns:
        kw_rows.append(kruskal_by_cluster_robust(master, "cluster_mod", v))

kw_df = pd.DataFrame(kw_rows)
if not kw_df.empty:
    kw_df["p_fmt"] = kw_df["p"].map(fmt_p)
kw_path = os.path.join(PART4_DIR, "kruskal_wallis_by_cluster_v3.csv")
kw_df.to_csv(kw_path, index=False)
print("✅ kruskal_wallis_by_cluster_v3.csv salvo:", kw_path)

# -----------------------------
# 8) Perfis por cluster (medianas úteis p/ narrativa)
# -----------------------------
profile_cols = ["cluster_mod"]
for c in [ies_col_nodes, uf_col_nodes, prog_col_nodes, nota_col_nodes]:
    if c is not None and c in master.columns:
        profile_cols.append(c)

agg_cols = [c for c in numeric_vars if c in master.columns]
cluster_profiles = (master
    .groupby("cluster_mod")[agg_cols]
    .agg(["count","median","mean"])
)

# flatten columns
cluster_profiles.columns = [f"{a}_{b}" for a,b in cluster_profiles.columns]
cluster_profiles = cluster_profiles.reset_index()

profiles_path = os.path.join(PART4_DIR, "cluster_profiles_v3.csv")
cluster_profiles.to_csv(profiles_path, index=False)
print("✅ cluster_profiles_v3.csv salvo:", profiles_path)

# -----------------------------
# 9) Hooks narrativos (JSON) — sem “inventar” tema, só “pistas”
# -----------------------------
hooks = {}
for _, row in cluster_profiles.iterrows():
    k = int(row["cluster_mod"])
    hooks[str(k)] = {
        "n": int(row.get("works_2010_count", 0)),
        "signals": {
            "degree_med": float(row.get("degree_median", 0.0)),
            "strength_med": float(row.get("strength_median", 0.0)),
            "works_2010_med": float(row.get("works_2010_median", 0.0)),
            "citations_2010_med": float(row.get("citations_2010_median", 0.0)),
            "active_years_med": float(row.get("active_years_since2010_median", 0.0)),
            "papers_ppgci_internal_med": float(row.get("papers_count_ppgci_internal_median", 0.0)),
            "cits_per_work_med": float(row.get("cits_per_work_median", 0.0)),
        }
    }

hooks_path = os.path.join(PART4_DIR, "cluster_narrative_hooks_v3.json")
with open(hooks_path, "w", encoding="utf-8") as f:
    json.dump(hooks, f, indent=2, ensure_ascii=False)
print("✅ cluster_narrative_hooks_v3.json salvo em:", hooks_path)

# -----------------------------
# 10) Prints tipo “paper”
# -----------------------------
def show_table(title, df):
    print("\n" + title)
    print("=" * len(title))
    print(df.to_string(index=False))

# Table IX (Kruskal) pronta para paper
if not kw_df.empty:
    show = kw_df.copy()
    show["H"] = show["H"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "NA")
    show["p_fmt"] = show["p_fmt"].fillna("NA")
    show_table("Table IX — Between-cluster differences (Kruskal–Wallis) [ROBUST]", show[["value","H","p_fmt","k_groups","note"]])

# Highlights simples
largest = master["cluster_mod"].value_counts().idxmax()
print("\nCluster profile highlights")
print("=========================")
print("Largest cluster:", int(largest))

# Top-5 por median WORKS e CITATIONS
if "works_2010" in master.columns:
    topW = (master.groupby("cluster_mod")
            .agg(n=("nome_tokensort_norm","count"),
                 works_2010_med=("works_2010","median"),
                 citations_2010_med=("citations_2010","median"),
                 degree_med=("degree","median"),
                 strength_med=("strength","median"))
            .sort_values("works_2010_med", ascending=False)
            .head(5)
            .reset_index())
    show_table("Top-5 clusters by median WORKS (2010+)", topW)

if "citations_2010" in master.columns:
    topC = (master.groupby("cluster_mod")
            .agg(n=("nome_tokensort_norm","count"),
                 works_2010_med=("works_2010","median"),
                 citations_2010_med=("citations_2010","median"),
                 degree_med=("degree","median"),
                 strength_med=("strength","median"))
            .sort_values("citations_2010_med", ascending=False)
            .head(5)
            .reset_index())
    show_table("Top-5 clusters by median CITATIONS (2010+)", topC)

print("\n====================")
print("Outputs saved in:", PART4_DIR)
print("====================")
for fn in [
    "master_node_table_v3.csv",
    "cluster_associations_v3.csv",
    "top_ies_by_cluster_v3.csv",
    "kruskal_wallis_by_cluster_v3.csv",
    "cluster_profiles_v3.csv",
    "cluster_narrative_hooks_v3.json"
]:
    p = os.path.join(PART4_DIR, fn)
    if os.path.exists(p):
        print(" -", p)


PARTE 4 (V5) — inputs:
 - nodes: (404, 4) | edges: (1490, 3)
 - K: 16 | N: 404
Sociotech cols (detectadas): {'UF': None, 'IES': 'SG_ENTIDADE_ENSINO', 'PROGRAMA': None, 'NOTA': None}
✅ producoes_final: producoes_final.xlsx | shape: (38138, 9)
✅ rebuilt produtividade (2010+): (404, 6) | via coluna autores (nome_tokensort_norm)
✅ master_node_table_v3.csv salvo: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/master_node_table_v3.csv
Master shape: (404, 13)
✅ Associação (cluster ~ IES): V = 0.621 | p = 2.1e-292
✅ cluster_associations_v3.csv salvo: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/cluster_associations_v3.csv
✅ top_ies_by_cluster_v3.csv salvo: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/top_ies_by_cluster_v3.csv
✅ kruskal_wallis_by_cluster_v3.csv salvo: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/kruskal_wallis_by_cluster_v3.csv
✅ cluster_profiles

In [ ]:
# ============================================================
# PARTE 4.B — Robustez: produtividade INTRA-PPGCI ONLY
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import kruskal

P4 = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4"

master = pd.read_csv(f"{P4}/master_node_table_v3.csv")

assert "papers_count_ppgci_internal" in master.columns, \
    "papers_count_ppgci_internal não encontrado."

def kruskal_safe(df, cluster_col, value_col):
    vals = df[value_col].dropna()
    if vals.nunique() <= 1:
        return {
            "value": value_col,
            "H": np.nan,
            "p": np.nan,
            "k_groups": 0,
            "note": "constant_global"
        }

    groups = [
        g[value_col].dropna().values
        for _, g in df.groupby(cluster_col)
        if g[value_col].dropna().nunique() > 0
    ]

    if len(groups) < 2:
        return {
            "value": value_col,
            "H": np.nan,
            "p": np.nan,
            "k_groups": len(groups),
            "note": "single_group"
        }

    H, p = kruskal(*groups)
    return {
        "value": value_col,
        "H": H,
        "p": p,
        "k_groups": len(groups),
        "note": "ok"
    }

row = kruskal_safe(
    master,
    cluster_col="cluster_mod",
    value_col="papers_count_ppgci_internal"
)

df_b = pd.DataFrame([row])
df_b["p_fmt"] = df_b["p"].apply(
    lambda x: "NA" if pd.isna(x) else f"{x:.1e}"
)

df_b.to_csv(f"{P4}/kruskal_intra_ppgci_only_v3.csv", index=False)

print("\nPARTE 4.B — Kruskal (INTRA-PPGCI ONLY)")
print(df_b[["value", "H", "p_fmt", "k_groups", "note"]].to_string(index=False))




PARTE 4.B — Kruskal (INTRA-PPGCI ONLY)
                      value         H   p_fmt  k_groups note
papers_count_ppgci_internal 62.264263 1.0e-07        16   ok


In [ ]:
# ============================================================
# PARTE 4.C — Sensibilidade: total vs intra-PPGCI
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

P4 = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4"

master = pd.read_csv(f"{P4}/master_node_table_v3.csv")

required = [
    "cluster_mod",
    "works_2010",
    "citations_2010",
    "papers_count_ppgci_internal"
]

for c in required:
    if c not in master.columns:
        raise KeyError(f"{c} ausente em master_node_table_v3.csv")

# -----------------------------
# Correlações individuais
# -----------------------------
rho_w, p_w = spearmanr(
    master["works_2010"],
    master["papers_count_ppgci_internal"],
    nan_policy="omit"
)

rho_c, p_c = spearmanr(
    master["citations_2010"],
    master["papers_count_ppgci_internal"],
    nan_policy="omit"
)

corr_df = pd.DataFrame({
    "metric_pair": [
        "works_2010 ~ papers_count_ppgci_internal",
        "citations_2010 ~ papers_count_ppgci_internal"
    ],
    "spearman_rho": [rho_w, rho_c],
    "p_value": [p_w, p_c]
})

corr_df.to_csv(f"{P4}/correlation_total_vs_intra_v3.csv", index=False)

# -----------------------------
# Rankings de clusters
# -----------------------------
cluster_rank = (
    master
    .groupby("cluster_mod")
    .agg(
        works_med=("works_2010", "median"),
        intra_med=("papers_count_ppgci_internal", "median")
    )
    .reset_index()
)

cluster_rank["rank_total"] = cluster_rank["works_med"].rank(
    ascending=False, method="average"
)
cluster_rank["rank_intra"] = cluster_rank["intra_med"].rank(
    ascending=False, method="average"
)

rho_rank, p_rank = spearmanr(
    cluster_rank["rank_total"],
    cluster_rank["rank_intra"]
)

cluster_rank["rank_diff"] = (
    cluster_rank["rank_total"] - cluster_rank["rank_intra"]
)

cluster_rank.to_csv(
    f"{P4}/cluster_rank_sensitivity_v3.csv",
    index=False
)

print("\nPARTE 4.C — Sensibilidade")
print("Correlação total vs intra:")
print(corr_df.to_string(index=False))

print("\nCorrelação entre rankings de clusters:")
print(f"Spearman rho = {rho_rank:.3f} | p = {p_rank:.2e}")

print("\nClusters com maior divergência de ranking:")
print(
    cluster_rank
    .sort_values("rank_diff", key=lambda x: abs(x), ascending=False)
    .head(5)
    .to_string(index=False)
)



PARTE 4.C — Sensibilidade
Correlação total vs intra:
                                 metric_pair  spearman_rho       p_value
    works_2010 ~ papers_count_ppgci_internal      0.853354 8.869309e-116
citations_2010 ~ papers_count_ppgci_internal      0.644415  8.846571e-49

Correlação entre rankings de clusters:
Spearman rho = 0.875 | p = 9.04e-06

Clusters com maior divergência de ranking:
 cluster_mod  works_med  intra_med  rank_total  rank_intra  rank_diff
           6       26.0       19.0        15.0         9.5        5.5
           2       41.0       13.5         8.0        12.0       -4.0
           8       30.5       19.5        11.0         8.0        3.0
          15       62.0       48.0         3.5         1.0        2.5
          14       64.0       38.5         1.5         4.0       -2.5


In [ ]:
# ============================================================
# PARTE 5 — Paper-facing consolidation (FIXED)
# Respondendo às recomendações do orientador e Stanford
# ============================================================

import pandas as pd
import numpy as np
import os

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART5 = f"{BASE}/part5"

# ------------------------------------------------------------
# 5.0 — Garantir diretório (FIX DO ERRO)
# ------------------------------------------------------------
os.makedirs(PART5, exist_ok=True)

# ------------------------------------------------------------
# Inputs finais (artefatos congelados)
# ------------------------------------------------------------
nodes = pd.read_csv(f"{BASE}/part3/nodes_plus_clusters_v3.csv")
edges = pd.read_csv(f"{BASE}/part3/edges_beta_v3.csv")
master = pd.read_csv(f"{BASE}/part4/master_node_table_v3.csv")

# ------------------------------------------------------------
# 5.1 — Sanity checks (anti-leakage)
# ------------------------------------------------------------
assert "cluster_mod" in nodes.columns
assert nodes.shape[0] == master.shape[0], "Mismatch nodes vs master"

print("PARTE 5 — Sanity checks")
print(f"Nodes: {nodes.shape[0]} | Edges: {edges.shape[0]}")
print(f"Clusters (K): {nodes['cluster_mod'].nunique()}")

# ------------------------------------------------------------
# 5.2 — Community size & composition (Table I)
# ------------------------------------------------------------
cluster_sizes = (
    nodes
    .groupby("cluster_mod")
    .size()
    .reset_index(name="n_nodes")
    .sort_values("n_nodes", ascending=False)
)

cluster_sizes.to_csv(
    f"{PART5}/table_cluster_sizes.csv",
    index=False
)

# ------------------------------------------------------------
# 5.3 — Sociotechnical alignment (IES dominance)
# ------------------------------------------------------------
if "SG_ENTIDADE_ENSINO" in master.columns:
    ies_dom = (
        master
        .groupby(["cluster_mod", "SG_ENTIDADE_ENSINO"])
        .size()
        .reset_index(name="n")
    )

    ies_top = (
        ies_dom
        .sort_values(["cluster_mod", "n"], ascending=[True, False])
        .groupby("cluster_mod")
        .head(1)
        .rename(columns={"SG_ENTIDADE_ENSINO": "top_IES"})
    )

    ies_top.to_csv(
        f"{PART5}/table_cluster_top_ies.csv",
        index=False
    )

# ------------------------------------------------------------
# 5.4 — Scientific differentiation (Table II)
# ------------------------------------------------------------
prod_cols = [
    "works_2010",
    "citations_2010",
    "papers_count_ppgci_internal",
    "degree",
    "strength"
]

prod_cols = [c for c in prod_cols if c in master.columns]

cluster_profiles = (
    master
    .groupby("cluster_mod")[prod_cols]
    .median()
    .reset_index()
)

cluster_profiles.to_csv(
    f"{PART5}/table_cluster_profiles.csv",
    index=False
)

# ------------------------------------------------------------
# 5.5 — Narrative hooks (qualitative bridge)
# ------------------------------------------------------------
hooks = []

for _, row in cluster_profiles.iterrows():
    hooks.append({
        "cluster": int(row["cluster_mod"]),
        "hook": (
            f"Cluster {int(row['cluster_mod'])} apresenta "
            f"mediana de {row.get('works_2010', np.nan):.0f} trabalhos (2010+), "
            f"{row.get('citations_2010', np.nan):.0f} citações e "
            f"centralidade média (strength) de {row.get('strength', np.nan):.1f}, "
            "indicando um padrão consistente de organização intra-campo."
        )
    })

hooks_df = pd.DataFrame(hooks)

hooks_df.to_csv(
    f"{PART5}/cluster_narrative_hooks_final.csv",
    index=False
)

print("\nPARTE 5 concluída com sucesso.")
print("Artefatos salvos em:", PART5)
print(" - table_cluster_sizes.csv")
print(" - table_cluster_top_ies.csv")
print(" - table_cluster_profiles.csv")
print(" - cluster_narrative_hooks_final.csv")


PARTE 5 — Sanity checks
Nodes: 404 | Edges: 1490
Clusters (K): 16

PARTE 5 concluída com sucesso.
Artefatos salvos em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part5
 - table_cluster_sizes.csv
 - table_cluster_top_ies.csv
 - table_cluster_profiles.csv
 - cluster_narrative_hooks_final.csv


In [ ]:
import pandas as pd

PART5 = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part5"

print("\n=== Table: Cluster sizes ===")
cs = pd.read_csv(f"{PART5}/table_cluster_sizes.csv")
print(cs.head(16).to_string(index=False))

print("\n=== Table: Top IES by cluster ===")
try:
    top = pd.read_csv(f"{PART5}/table_cluster_top_ies.csv")
    print(top.sort_values("cluster_mod").to_string(index=False))
except FileNotFoundError:
    print("(arquivo não gerado — SG_ENTIDADE_ENSINO não estava no master)")

print("\n=== Table: Cluster profiles (median) ===")
cp = pd.read_csv(f"{PART5}/table_cluster_profiles.csv")
print(cp.sort_values("cluster_mod").to_string(index=False))

print("\n=== Narrative hooks (preview) ===")
hooks = pd.read_csv(f"{PART5}/cluster_narrative_hooks_final.csv")
print(hooks.head(8).to_string(index=False))



=== Table: Cluster sizes ===
 cluster_mod  n_nodes
          10       44
           0       42
          11       39
           4       34
           6       29
           8       28
           7       27
           5       27
          13       25
           2       24
           1       19
           3       19
           9       19
          12       13
          14       12
          15        3

=== Table: Top IES by cluster ===
 cluster_mod   top_IES  n
           0    UNIRIO 15
           1 UNESP-MAR  7
           2     FUMEC 15
           3      UFES  9
           4     UDESC 15
           5   UFPB-JP 12
           6      UFMG 23
           7       UFF 11
           8      UFPE  8
           9 UNESP-MAR  7
          10       UFC 12
          11    UNIRIO  7
          12     FUFSE  7
          13       UEL 11
          14     UFRGS 11
          15   UFPB-JP  2

=== Table: Cluster profiles (median) ===
 cluster_mod  works_2010  citations_2010  papers_count_ppgci_internal  degree

In [ ]:
import pandas as pd
import os

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"

TABLES = {
    # --- Resultados principais ---
    "Table 1 — Network summary": f"{BASE}/part1/network_summary.csv",
    "Table 2 — Backbone audit (beta)": f"{BASE}/part2/backbone_audit.csv",
    "Table 3 — Community sizes": f"{BASE}/part5/table_cluster_sizes.csv",
    "Table 4 — Cluster × IES association": f"{BASE}/part4/cluster_associations_v3.csv",

    # --- Perfis e diferenças ---
    "Table 9 — Kruskal-Wallis (clusters)": f"{BASE}/part4/kruskal_wallis_by_cluster_v3.csv",
    "Table — Cluster profiles (medians)": f"{BASE}/part5/table_cluster_profiles.csv",

    # --- Robustez ---
    "Table — Conductance by cluster": f"{BASE}/part3/conductance_by_cluster_v3.csv",
    "Table — Bootstrap stability": f"{BASE}/part3/bootstrap_stability_v3.csv",

    # --- Sensibilidade ---
    "Table — Intra-PPGCI only (4.B)": f"{BASE}/part4/kruskal_intra_ppgci_only_v3.csv",
    "Table — Rank sensitivity (4.C)": f"{BASE}/part4/cluster_rank_sensitivity_v3.csv",
}

for title, path in TABLES.items():
    print("\n" + "="*len(title))
    print(title)
    print("="*len(title))
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(df.to_string(index=False))
    else:
        print(f"[Arquivo não encontrado: {path}]")



Table 1 — Network summary
[Arquivo não encontrado: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part1/network_summary.csv]

Table 2 — Backbone audit (beta)
 beta    algo  n_nodes  n_edges  n_components  giant_size  K   H_norm  modularity_Q  coverage  conductance_mean  conductance_weighted_mean  asymptotic_surprise  V_UF  p_UF    V_IES         p_IES  seconds
 0.25 louvain      404      433            82         272 96 0.820118      0.841796  0.929425          0.011575                   0.070575          1087.761215   NaN   NaN 0.666476 1.316955e-122 1.022294
 0.50 louvain      404      987            12         383 26 0.838802      0.752474  0.847692          0.088945                   0.152308          1311.238608   NaN   NaN 0.480931 5.333772e-180 0.228234
 0.75 louvain      404     1490             1         404 11 0.958399      0.708143  0.809332          0.188377                   0.190668          1254.456376   NaN   NaN 0.654375 6.603409e-228 0.282726

In [ ]:
BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
import os
assert os.path.exists(BASE), f"BASE não existe: {BASE}"
print("✅ BASE OK:", BASE)
print("Conteúdo:", sorted(os.listdir(BASE))[:20])


✅ BASE OK: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407
Conteúdo: ['part6']


In [ ]:
import os

needed = [
    f"{BASE}/part3/nodes_plus_clusters_v3.csv",
    f"{BASE}/part3/edges_beta_v3.csv",
    f"{BASE}/part3/paper_ready_selection_audit.csv",
    f"{BASE}/part3/conductance_by_cluster_v3.csv",
    f"{BASE}/part3/bootstrap_stability_v3.csv",
    f"{BASE}/part3/bootstrap_persistence_v3.json",
    f"{BASE}/part4/master_node_table_v3.csv",
    f"{BASE}/part4/cluster_associations_v3.csv",
    f"{BASE}/part4/kruskal_wallis_by_cluster_v3.csv",
    f"{BASE}/part5/table_cluster_sizes.csv",
    f"{BASE}/part5/table_cluster_top_ies.csv",
    f"{BASE}/part5/table_cluster_profiles.csv",
]

missing = [p for p in needed if not os.path.exists(p)]
print("Faltando:", missing if missing else "✅ nada")


Faltando: ['/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/nodes_plus_clusters_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/edges_beta_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/paper_ready_selection_audit.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/conductance_by_cluster_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/bootstrap_stability_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/bootstrap_persistence_v3.json', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/master_node_table_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/cluster_associations_v3.csv', '/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/kruskal_wallis_by_cluster_v3.csv', '

In [ ]:
PART6 = f"{BASE}/part6"
os.makedirs(PART6, exist_ok=True)
print("✅ PART6:", PART6)


✅ PART6: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6


In [ ]:
import os, glob

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
print("BASE exists?", os.path.exists(BASE))
print("BASE contents:\n", "\n".join(sorted(os.listdir(BASE))))

PART3 = f"{BASE}/part3"
print("\nPART3 exists?", os.path.exists(PART3))
if os.path.exists(PART3):
    print("PART3 files:\n", "\n".join(sorted(os.listdir(PART3))))


BASE exists? True
BASE contents:
 part6

PART3 exists? False


In [ ]:
import os, glob

RUNS_DIR = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs"
print("RUNS_DIR exists?", os.path.exists(RUNS_DIR))

runs = sorted([d for d in os.listdir(RUNS_DIR) if d.startswith("run_")])
print("Total runs:", len(runs))
print("Last 20 runs:", runs[-20:])

candidates = []
for r in runs:
    base = os.path.join(RUNS_DIR, r)
    p3 = os.path.join(base, "part3", "nodes_plus_clusters_v3.csv")
    p4 = os.path.join(base, "part4", "master_node_table_v3.csv")
    if os.path.exists(p3) and os.path.exists(p4):
        candidates.append(base)

print("\nRuns with BOTH part3+part4 artifacts:")
print("\n".join(candidates) if candidates else "❌ none found")


RUNS_DIR exists? True
Total runs: 1
Last 20 runs: ['run_20260202_175407']

Runs with BOTH part3+part4 artifacts:
❌ none found


In [ ]:
import glob

ROOT = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"

hits_nodes = sorted(glob.glob(ROOT + "/**/nodes_plus_clusters_v3.csv", recursive=True))
hits_edges = sorted(glob.glob(ROOT + "/**/edges_beta_v3.csv", recursive=True))
hits_master = sorted(glob.glob(ROOT + "/**/master_node_table_v3.csv", recursive=True))

print("nodes_plus_clusters_v3.csv:", len(hits_nodes))
print("\n".join(hits_nodes[:50]) if hits_nodes else "❌ none")

print("\nedges_beta_v3.csv:", len(hits_edges))
print("\n".join(hits_edges[:50]) if hits_edges else "❌ none")

print("\nmaster_node_table_v3.csv:", len(hits_master))
print("\n".join(hits_master[:50]) if hits_master else "❌ none")


nodes_plus_clusters_v3.csv: 0
❌ none

edges_beta_v3.csv: 0
❌ none

master_node_table_v3.csv: 0
❌ none


In [ ]:
import glob

hits = []
for base in ["/content", "/tmp"]:
    hits += glob.glob(base + "/**/nodes_plus_clusters_v3.csv", recursive=True)
    hits += glob.glob(base + "/**/edges_beta_v3.csv", recursive=True)
    hits += glob.glob(base + "/**/master_node_table_v3.csv", recursive=True)

hits = sorted(set(hits))
print("Total hits:", len(hits))
print("\n".join(hits[:200]) if hits else "❌ none")


Total hits: 0
❌ none


In [ ]:
import os

DRIVE_ROOT = "/content/drive/MyDrive"
targets = {
    "producoes_final.xlsx",
    "autores_final.xlsx",
    "banco_docente-PPGCI.xlsx",
    "banco-sucupira_docente-PPGCI.xlsx",
    "banco_sucupira_docente-PPGCI.xlsx",
}

hits = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    for f in files:
        if f in targets or f.lower() in {t.lower() for t in targets}:
            hits.append(os.path.join(root, f))

hits = sorted(hits)
print("Found:", len(hits))
for p in hits[:200]:
    print(p)


Found: 0


In [ ]:
import os, glob

print("Exists /content/drive ?", os.path.exists("/content/drive"))
print("Exists /content/drive/MyDrive ?", os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive"):
    print("\nTop of /content/drive:")
    print(os.listdir("/content/drive")[:50])

if os.path.exists("/content/drive/MyDrive"):
    print("\nMyDrive top-level (first 50):")
    print(os.listdir("/content/drive/MyDrive")[:50])

print("\nCount of XLSX in MyDrive (recursive, may be slow but ok):")
xlsx = glob.glob("/content/drive/MyDrive/**/*.xlsx", recursive=True)
print("xlsx:", len(xlsx))
print("sample:", xlsx[:20])

print("\nCount of CSV in MyDrive (recursive):")
csv = glob.glob("/content/drive/MyDrive/**/*.csv", recursive=True)
print("csv:", len(csv))
print("sample:", csv[:20])


Exists /content/drive ? True
Exists /content/drive/MyDrive ? True

Top of /content/drive:
['MyDrive']

MyDrive top-level (first 50):
['PPGCI_intra_field_lab_v2']

Count of XLSX in MyDrive (recursive, may be slow but ok):
xlsx: 0
sample: []

Count of CSV in MyDrive (recursive):
csv: 0
sample: []


In [ ]:
import os, glob

ROOT = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2"
RUN  = f"{ROOT}/runs/run_20260202_175407"

def ls(p):
    print(f"\n== {p} ==")
    if not os.path.exists(p):
        print("NOT FOUND")
        return
    items = sorted(os.listdir(p))
    print("n_items:", len(items))
    print("first 200:", items[:200])

ls(ROOT)
ls(f"{ROOT}/runs")
ls(RUN)

# lista pastas part* e mostra o que tem em cada uma
parts = sorted([p for p in glob.glob(f"{RUN}/part*") if os.path.isdir(p)])
print("\nPART directories:", parts)

for p in parts:
    ls(p)

# quais extensões existem nesse RUN?
all_files = [f for f in glob.glob(f"{RUN}/**/*", recursive=True) if os.path.isfile(f)]
print("\nTotal files under RUN:", len(all_files))
exts = {}
for f in all_files:
    ext = os.path.splitext(f)[1].lower() or "(noext)"
    exts[ext] = exts.get(ext, 0) + 1
print("\nExtensions under RUN (sorted):")
for k in sorted(exts, key=lambda x: (-exts[x], x)):
    print(k, ":", exts[k])

print("\nSample files (first 60):")
print(all_files[:60])



== /content/drive/MyDrive/PPGCI_intra_field_lab_v2 ==
n_items: 1
first 200: ['runs']

== /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs ==
n_items: 1
first 200: ['run_20260202_175407']

== /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407 ==
n_items: 1
first 200: ['part6']

PART directories: ['/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6']

== /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6 ==
n_items: 0
first 200: []

Total files under RUN: 0

Extensions under RUN (sorted):

Sample files (first 60):
[]


In [ ]:
import os, glob, shutil

# Onde deveria estar no Drive
DRIVE_RUN = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"

# O que estamos procurando (arquivos-chave)
targets = [
    "nodes_plus_clusters_v3.csv",
    "edges_beta_v3.csv",
    "master_node_table_v3.csv",
    "producoes_final.xlsx",
    "autores_final.xlsx",
]

# 1) Buscar no runtime do Colab
roots = ["/content", "/mnt/data", "/tmp"]
found = []

for r in roots:
    if not os.path.exists(r):
        continue
    for t in targets:
        hits = glob.glob(f"{r}/**/{t}", recursive=True)
        for h in hits:
            if os.path.isfile(h):
                found.append(h)

found = sorted(set(found))

print("Found files:", len(found))
for f in found[:200]:
    print(" -", f)

# 2) Se achar, copiar para a estrutura correta do Drive (part3/part4/input)
#    (não copia automaticamente; só prepara a ação e mostra o plano)
plan = []
def plan_copy(src):
    name = os.path.basename(src)
    if name in ["nodes_plus_clusters_v3.csv", "edges_beta_v3.csv"]:
        dst_dir = os.path.join(DRIVE_RUN, "part3")
    elif name == "master_node_table_v3.csv":
        dst_dir = os.path.join(DRIVE_RUN, "part4")
    elif name.endswith(".xlsx"):
        dst_dir = os.path.join(DRIVE_RUN, "input")
    else:
        dst_dir = DRIVE_RUN
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, name)
    plan.append((src, dst))

for src in found:
    if os.path.basename(src) in targets:
        plan_copy(src)

print("\nCopy plan (src -> dst):", len(plan))
for src, dst in plan:
    print(" -", src, "->", dst)

# 3) Execute a cópia (APENAS se houver plano)
if plan:
    for src, dst in plan:
        shutil.copy2(src, dst)
    print("\n✅ Copied all planned files to Drive run folder.")
    print("Drive run contents now:", os.listdir(DRIVE_RUN))
else:
    print("\n❌ Nothing to copy. No target artifacts found in runtime paths.")


Found files: 0

Copy plan (src -> dst): 0

❌ Nothing to copy. No target artifacts found in runtime paths.


In [ ]:
import os, glob

# 1) Verificar montagem e identificar "quem" é o MyDrive (pelo menos listar muita coisa)
print("Exists /content/drive/MyDrive ?", os.path.exists("/content/drive/MyDrive"))

# 2) Contagem de arquivos gerais no MyDrive (não só no seu projeto)
# (pega até 2000 paths para não explodir)
all_files = glob.glob("/content/drive/MyDrive/**", recursive=True)
all_files = [p for p in all_files if os.path.isfile(p)]
print("Total files in MyDrive (recursive):", len(all_files))
print("Sample (first 40):")
for p in all_files[:40]:
    print(" -", p)

# 3) Contagem por extensão (rápida)
ext_counts = {}
for p in all_files:
    ext = os.path.splitext(p)[1].lower()
    ext_counts[ext] = ext_counts.get(ext, 0) + 1

print("\nTop extensions:")
for k,v in sorted(ext_counts.items(), key=lambda x: x[1], reverse=True)[:15]:
    print(k, v)

# 4) Procurar qualquer coisa relacionada ao seu lab (nome do projeto, runs, outputs)
keywords = ["PPGCI", "intra_field", "run_", "nodes_plus", "edges_beta", "master_node", "producoes_final", "autores_final"]
hits = []
for kw in keywords:
    hh = glob.glob(f"/content/drive/MyDrive/**/*{kw}*", recursive=True)
    hh = [p for p in hh if os.path.isfile(p)]
    hits += hh

hits = sorted(set(hits))
print("\nKeyword hits:", len(hits))
for p in hits[:80]:
    print(" -", p)


Exists /content/drive/MyDrive ? True
Total files in MyDrive (recursive): 0
Sample (first 40):

Top extensions:

Keyword hits: 0


In [ ]:
import os, time, glob

SENT_DIR = "/content/drive/MyDrive/__TESTE_COLAB__"
os.makedirs(SENT_DIR, exist_ok=True)

fname = f"sentinela_{int(time.time())}.txt"
fpath = os.path.join(SENT_DIR, fname)

with open(fpath, "w", encoding="utf-8") as f:
    f.write("se você está lendo isso, o Colab escreveu no seu MyDrive.\n")

print("✅ Arquivo criado no MyDrive:", fpath)
print("Conteúdo da pasta __TESTE_COLAB__:", os.listdir(SENT_DIR)[:10])

# checagem: quantos arquivos existem no MyDrive (rápida, só 2 níveis)
top = "/content/drive/MyDrive"
lvl1 = glob.glob(top+"/*")
lvl2 = []
for p in lvl1[:200]:
    if os.path.isdir(p):
        lvl2 += glob.glob(p+"/*")
print("\nMyDrive nível 1 (amostra):", [os.path.basename(x) for x in lvl1[:20]])
print("Total itens nível 1:", len(lvl1))
print("Total itens nível 2 (amostra):", len(lvl2))



✅ Arquivo criado no MyDrive: /content/drive/MyDrive/__TESTE_COLAB__/sentinela_1770131604.txt
Conteúdo da pasta __TESTE_COLAB__: ['sentinela_1770131604.txt']

MyDrive nível 1 (amostra): ['__TESTE_COLAB__', 'PPGCI_intra_field_lab_v2']
Total itens nível 1: 2
Total itens nível 2 (amostra): 2


In [ ]:
import os, glob

candidates = glob.glob("/content/*.zip")
print("Zips em /content:", len(candidates))
print("Lista:", [os.path.basename(p) for p in candidates])

if candidates:
    ZIP_PATH = max(candidates, key=os.path.getmtime)
    print("\n✅ ZIP_PATH selecionado:", ZIP_PATH)


Zips em /content: 0
Lista: []


In [ ]:
from google.colab import drive
import os, shutil

# 1) desmonta (se estiver montado)
try:
    drive.flush_and_unmount()
except Exception as e:
    print("flush_and_unmount() avisou:", e)

# 2) remove o diretório de mount inteiro (é isso que resolve o 'mountpoint not empty')
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)

# 3) cria de novo vazio e monta com force_remount
os.makedirs("/content/drive", exist_ok=True)
drive.mount("/content/drive", force_remount=True)


Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
import os

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
print("BASE exists?", os.path.exists(BASE))
print("Conteúdo de BASE:", os.listdir(BASE) if os.path.exists(BASE) else None)

# checar especificamente part3/part4/part5
for p in ["part3","part4","part5","input","paper_outputs"]:
    pp = os.path.join(BASE, p)
    print(p, "->", os.path.exists(pp), "| itens:", (len(os.listdir(pp)) if os.path.exists(pp) else 0))


BASE exists? True
Conteúdo de BASE: ['input', 'intermediate', 'tables', 'figures', 'models', 'logs', 'artifacts', 'part2', 'part3', 'part4', 'part5', 'paper_tables_appendix.txt', 'paper_outputs']
part3 -> True | itens: 8
part4 -> True | itens: 9
part5 -> True | itens: 4
input -> True | itens: 6
paper_outputs -> True | itens: 1


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"

paths = {
    "nodes":  f"{BASE}/part3/nodes_plus_clusters_v3.csv",
    "edges":  f"{BASE}/part3/edges_beta_v3.csv",
    "master": f"{BASE}/part4/master_node_table_v3.csv",
}

for k, p in paths.items():
    print(k, "exists?", os.path.exists(p), "|", p)

# carregar (amostra)
nodes  = pd.read_csv(paths["nodes"])
edges  = pd.read_csv(paths["edges"])
master = pd.read_csv(paths["master"])

print("\nShapes:")
print("nodes :", nodes.shape, "| cols:", list(nodes.columns)[:10])
print("edges :", edges.shape, "| cols:", list(edges.columns)[:10])
print("master:", master.shape, "| cols:", list(master.columns)[:15])

# sanity checks que evitam dor de cabeça depois
assert "cluster_mod" in nodes.columns, "nodes sem cluster_mod"
assert nodes.shape[0] == master.shape[0], "Mismatch nodes vs master"
print("\n✅ Sanity checks OK")


nodes exists? True | /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/nodes_plus_clusters_v3.csv
edges exists? True | /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part3/edges_beta_v3.csv
master exists? True | /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part4/master_node_table_v3.csv

Shapes:
nodes : (404, 4) | cols: ['nome_tokensort_norm', 'instituicao_original', 'SG_ENTIDADE_ENSINO', 'cluster_mod']
edges : (1490, 3) | cols: ['u', 'v', 'w']
master: (404, 13) | cols: ['nome_tokensort_norm', 'cluster_mod', 'SG_ENTIDADE_ENSINO', 'degree', 'strength', 'avg_tie_strength', 'log1p_degree', 'log1p_strength', 'works_2010', 'citations_2010', 'active_years_since2010', 'papers_count_ppgci_internal', 'cits_per_work']

✅ Sanity checks OK


In [ ]:
import os, math, json
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"
INPUT = f"{BASE}/input"
os.makedirs(PART6, exist_ok=True)

# -----------------------------
# carregar artefatos (já ok)
# -----------------------------
nodes  = pd.read_csv(f"{BASE}/part3/nodes_plus_clusters_v3.csv")
edges  = pd.read_csv(f"{BASE}/part3/edges_beta_v3.csv")
master = pd.read_csv(f"{BASE}/part4/master_node_table_v3.csv")

# -----------------------------
# helpers
# -----------------------------
def fmt_p(p):
    if pd.isna(p): return "NA"
    if p == 0: return "0"
    if p < 1e-300: return "<1e-300"
    if p < 1e-3: return f"{p:.1e}"
    return f"{p:.4f}"

def cramers_v(x, y):
    # Cramér's V via chi2_contingency
    from scipy.stats import chi2_contingency
    ct = pd.crosstab(x, y)
    if ct.size == 0 or ct.shape[0] < 2 or ct.shape[1] < 2:
        return (np.nan, np.nan, np.nan, ct.shape)
    chi2, p, dof, exp = chi2_contingency(ct, correction=False)
    n = ct.to_numpy().sum()
    r, k = ct.shape
    denom = n * (min(r, k) - 1)
    V = math.sqrt(chi2 / denom) if denom > 0 else np.nan
    return (V, chi2, p, ct.shape)

def safe_quantiles(s, qs=(0.75,0.90)):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return {q: np.nan for q in qs}
    return {q: float(s.quantile(q)) for q in qs}

# -----------------------------
# (A) TABLE 1 — Cohort/Network Processing Summary (do que existe agora)
# -----------------------------
# Observação: no run novo você já está no recorte intra-PPGCI GIANT (404).
# Alguns números "pré-filter 84291" etc. não estão em arquivos atuais; então a Table 1 nova
# vai refletir o pipeline atual (producoes_final + backbone + intra-network).
# Se você quiser preservar os números antigos, a gente injeta depois manualmente (sem problema).

# works processed = total linhas em producoes_final
prod_path = None
for cand in ["producoes_final.xlsx", "producoes_final (1).xlsx", "producoes_final (2).xlsx"]:
    p = os.path.join(INPUT, cand)
    if os.path.exists(p):
        prod_path = p
        break
if prod_path is None:
    raise FileNotFoundError("Não encontrei producoes_final.xlsx dentro de input/")

producoes = pd.read_excel(prod_path)

table1 = pd.DataFrame([
    ["Works processed (rows in producoes_final)", int(producoes.shape[0])],
    ["Nodes in intra-PPGCI network (this run)", int(nodes.shape[0])],
    ["Edges in intra-PPGCI network (this run)", int(edges.shape[0])],
    ["Communities (K)", int(nodes["cluster_mod"].nunique())],
], columns=["Metric","Value"])

table1.to_csv(f"{PART6}/table1_processing_summary.csv", index=False)

# -----------------------------
# (B) TABLE 2 — Backbone Audit (já existe em part3: paper_ready_selection_audit.csv)
# -----------------------------
audit_path = f"{BASE}/part3/paper_ready_selection_audit.csv"
if os.path.exists(audit_path):
    audit = pd.read_csv(audit_path)
    # ordenar por score desc e salvar top (para paper)
    audit_sorted = audit.sort_values("score", ascending=False).reset_index(drop=True)
    audit_sorted.to_csv(f"{PART6}/table2_backbone_audit_full.csv", index=False)
    audit_top = audit_sorted.head(10)
    audit_top.to_csv(f"{PART6}/table2_backbone_audit_top10.csv", index=False)
else:
    audit_sorted = None

# -----------------------------
# (C) TABLE 3 — Community size distribution
# -----------------------------
table3 = (
    nodes.groupby("cluster_mod").size()
    .reset_index(name="size")
    .sort_values("size", ascending=False)
)
table3.to_csv(f"{PART6}/table3_cluster_sizes.csv", index=False)

# -----------------------------
# (D) TABLE 4 — Associations between cluster and exogenous metadata (IES only here)
# -----------------------------
assoc_rows = []
if "SG_ENTIDADE_ENSINO" in master.columns:
    V, chi2, p, shape = cramers_v(master["cluster_mod"], master["SG_ENTIDADE_ENSINO"])
    assoc_rows.append({
        "Pair": "IES × cluster",
        "Cramers_V": V,
        "chi2": chi2,
        "p_value": p,
        "p_fmt": fmt_p(p),
        "Dimensions": f"{shape[0]}×{shape[1]}"
    })
else:
    assoc_rows.append({"Pair":"IES × cluster","Cramers_V":np.nan,"chi2":np.nan,"p_value":np.nan,"p_fmt":"NA","Dimensions":"NA"})

table4 = pd.DataFrame(assoc_rows)
table4.to_csv(f"{PART6}/table4_associations.csv", index=False)

# -----------------------------
# (E) TABLE 7–8 equivalent (IES-level summaries for works/citations 2010+)
#     (porque UF não existe no run atual)
# -----------------------------
ies = "SG_ENTIDADE_ENSINO"
if ies in master.columns:
    # WORKS
    t7 = []
    for g, sub in master.groupby(ies):
        n = sub.shape[0]
        med = float(np.median(sub["works_2010"]))
        mean = float(np.mean(sub["works_2010"]))
        qs = safe_quantiles(sub["works_2010"], (0.75,0.90))
        t7.append([g, n, med, mean, qs[0.75], qs[0.90]])
    table7 = pd.DataFrame(t7, columns=["IES","n","median","mean","p75","p90"]).sort_values(["median","mean"], ascending=False)
    table7.to_csv(f"{PART6}/table7_ies_works_summary.csv", index=False)

    # CITATIONS
    t8 = []
    for g, sub in master.groupby(ies):
        n = sub.shape[0]
        med = float(np.median(sub["citations_2010"]))
        mean = float(np.mean(sub["citations_2010"]))
        qs = safe_quantiles(sub["citations_2010"], (0.75,0.90))
        t8.append([g, n, med, mean, qs[0.75], qs[0.90]])
    table8 = pd.DataFrame(t8, columns=["IES","n","median","mean","p75","p90"]).sort_values(["median","mean"], ascending=False)
    table8.to_csv(f"{PART6}/table8_ies_citations_summary.csv", index=False)

# -----------------------------
# (F) TABLE 9 — Kruskal-Wallis between-cluster differences (já calculado em part4)
# -----------------------------
kw_path = f"{BASE}/part4/kruskal_wallis_by_cluster_v3.csv"
if os.path.exists(kw_path):
    kw = pd.read_csv(kw_path)
    # padroniza nome para paper
    kw_out = kw.copy()
    kw_out["p_fmt"] = kw_out["p"].apply(lambda x: fmt_p(x))
    kw_out.to_csv(f"{PART6}/table9_kruskal_by_cluster.csv", index=False)
else:
    kw_out = None

# -----------------------------
# (G) TABLE 13–15 — Quality, conductance, stability, persistence (do part3)
# -----------------------------
# Table 13: community_metrics_overall_v3.json
cm_path = f"{BASE}/part3/community_metrics_overall_v3.json"
if os.path.exists(cm_path):
    with open(cm_path, "r", encoding="utf-8") as f:
        cm = json.load(f)
    table13 = pd.DataFrame([
        ["Nodes", cm.get("n_nodes")],
        ["Edges", cm.get("n_edges")],
        ["Communities (K)", cm.get("K")],
        ["Selected beta", cm.get("selected_beta")],
        ["Selected resolution", cm.get("selected_resolution")],
        ["Modularity (Q)", cm.get("Q")],
        ["Coverage", cm.get("coverage")],
        ["Conductance (mean)", cm.get("conductance_mean")],
        ["Conductance (weighted mean)", cm.get("conductance_weighted_mean")],
        ["Asymptotic Surprise (S)", cm.get("asymptotic_surprise")],
    ], columns=["Metric","Value"])
    table13.to_csv(f"{PART6}/table13_quality_metrics.csv", index=False)

# Table 14: conductance_by_cluster_v3.csv
cond_path = f"{BASE}/part3/conductance_by_cluster_v3.csv"
if os.path.exists(cond_path):
    cond = pd.read_csv(cond_path)
    cond.to_csv(f"{PART6}/table14_conductance_by_cluster.csv", index=False)

# Table 15: bootstrap_stability_v3.csv + bootstrap_persistence_v3.json
stab_path = f"{BASE}/part3/bootstrap_stability_v3.csv"
if os.path.exists(stab_path):
    stab = pd.read_csv(stab_path)
    stab.to_csv(f"{PART6}/table15_bootstrap_stability.csv", index=False)

pers_path = f"{BASE}/part3/bootstrap_persistence_v3.json"
if os.path.exists(pers_path):
    with open(pers_path, "r", encoding="utf-8") as f:
        pers = json.load(f)
    # pers pode ser dict; vamos “achatar” para csv
    rows = []
    # esperado: global + por cluster
    if isinstance(pers, dict):
        for k,v in pers.items():
            if isinstance(v, dict):
                row = {"key": k}
                row.update(v)
                rows.append(row)
            else:
                rows.append({"key": k, "value": v})
    pers_df = pd.DataFrame(rows)
    pers_df.to_csv(f"{PART6}/table15_pairwise_persistence.csv", index=False)

print("✅ PART6 gerado em:", PART6)
print("Arquivos criados (part6):")
print(sorted(os.listdir(PART6)))


✅ PART6 gerado em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6
Arquivos criados (part6):
['table13_quality_metrics.csv', 'table14_conductance_by_cluster.csv', 'table15_bootstrap_stability.csv', 'table15_pairwise_persistence.csv', 'table1_processing_summary.csv', 'table2_backbone_audit_full.csv', 'table2_backbone_audit_top10.csv', 'table3_cluster_sizes.csv', 'table4_associations.csv', 'table7_ies_works_summary.csv', 'table8_ies_citations_summary.csv', 'table9_kruskal_by_cluster.csv']


In [ ]:
import os
import numpy as np
import pandas as pd

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"

def fmt_num(x, nd=3):
    if pd.isna(x): return "NA"
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def fmt_int(x):
    if pd.isna(x): return "NA"
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x)

def fmt_p(p):
    if pd.isna(p): return "NA"
    p = float(p)
    if p == 0: return "0"
    if p < 1e-300: return "<1e-300"
    if p < 1e-3: return f"{p:.1e}"
    return f"{p:.4f}"

# -----------------------------
# carregar outputs do part6
# -----------------------------
t1  = pd.read_csv(f"{PART6}/table1_processing_summary.csv")
t3  = pd.read_csv(f"{PART6}/table3_cluster_sizes.csv")
t4  = pd.read_csv(f"{PART6}/table4_associations.csv")
t7  = pd.read_csv(f"{PART6}/table7_ies_works_summary.csv")
t8  = pd.read_csv(f"{PART6}/table8_ies_citations_summary.csv")
t9  = pd.read_csv(f"{PART6}/table9_kruskal_by_cluster.csv")

# Table 2 (audit top10) — pode usar como “Backbone audit”
t2 = pd.read_csv(f"{PART6}/table2_backbone_audit_top10.csv")

# Table 13–15
t13 = pd.read_csv(f"{PART6}/table13_quality_metrics.csv")
t14 = pd.read_csv(f"{PART6}/table14_conductance_by_cluster.csv")
t15 = pd.read_csv(f"{PART6}/table15_bootstrap_stability.csv")
t15p = pd.read_csv(f"{PART6}/table15_pairwise_persistence.csv")

# -----------------------------
# padronizações “paper-like”
# -----------------------------
# Table 2 — seleciona colunas equivalentes às do paper antigo
# (no seu run atual, UF não existe; fica V_IES e estabilidade NMI/Hnorm etc.)
cols2 = [c for c in ["beta","resolution","Q","V_IES","p_IES","H_norm","K","score"] if c in t2.columns]
t2_paper = t2[cols2].copy()
# arredondamentos
for c in ["beta","resolution","Q","V_IES","H_norm","score"]:
    if c in t2_paper.columns:
        t2_paper[c] = t2_paper[c].map(lambda x: fmt_num(x, 3))
if "p_IES" in t2_paper.columns:
    t2_paper["p_IES"] = t2_paper["p_IES"].map(fmt_p)
if "K" in t2_paper.columns:
    t2_paper["K"] = t2_paper["K"].map(fmt_int)

t2_paper.to_csv(f"{PART6}/table2_backbone_audit_paperlike.csv", index=False)

# Table 3 sizes
t3_paper = t3.copy()
t3_paper["cluster_mod"] = t3_paper["cluster_mod"].map(fmt_int)
t3_paper["size"] = t3_paper["size"].map(fmt_int)
t3_paper.to_csv(f"{PART6}/table3_cluster_sizes_paperlike.csv", index=False)

# Table 4 associations
t4_paper = t4.copy()
if "Cramers_V" in t4_paper.columns:
    t4_paper["Cramers_V"] = t4_paper["Cramers_V"].map(lambda x: fmt_num(x, 3))
if "chi2" in t4_paper.columns:
    t4_paper["chi2"] = t4_paper["chi2"].map(lambda x: fmt_num(x, 2))
if "p_value" in t4_paper.columns:
    t4_paper["p_value"] = t4_paper["p_value"].map(fmt_p)
t4_paper.to_csv(f"{PART6}/table4_associations_paperlike.csv", index=False)

# Table 7–8 IES summaries (ordem e arredondamento)
for df, outname in [(t7,"table7_ies_works_summary_paperlike.csv"),
                    (t8,"table8_ies_citations_summary_paperlike.csv")]:
    d = df.copy()
    d["n"] = d["n"].map(fmt_int)
    for c in ["median","mean","p75","p90"]:
        if c in d.columns:
            d[c] = d[c].map(lambda x: fmt_num(x, 1))
    d.to_csv(f"{PART6}/{outname}", index=False)

# Table 9 Kruskal
t9_paper = t9.copy()
if "H" in t9_paper.columns:
    t9_paper["H"] = t9_paper["H"].map(lambda x: fmt_num(x, 3))
if "p" in t9_paper.columns:
    t9_paper["p"] = t9_paper["p"].map(fmt_p)
t9_paper.to_csv(f"{PART6}/table9_kruskal_by_cluster_paperlike.csv", index=False)

# Table 13 quality
t13_paper = t13.copy()
t13_paper.to_csv(f"{PART6}/table13_quality_metrics_paperlike.csv", index=False)

# Table 14 conductance
t14_paper = t14.copy()
t14_paper.to_csv(f"{PART6}/table14_conductance_by_cluster_paperlike.csv", index=False)

# Table 15 stability + persistence
t15.to_csv(f"{PART6}/table15_bootstrap_stability_paperlike.csv", index=False)
t15p.to_csv(f"{PART6}/table15_pairwise_persistence_paperlike.csv", index=False)

# -----------------------------
# consolidar em um TXT único (apêndice)
# -----------------------------
def block(title, df, max_rows=40):
    s = []
    s.append(title)
    s.append("=" * len(title))
    if df.shape[0] > max_rows:
        s.append(df.head(max_rows).to_string(index=False))
        s.append(f"... (truncado em {max_rows} linhas; veja CSV completo em part6)")
    else:
        s.append(df.to_string(index=False))
    s.append("")  # linha em branco
    return "\n".join(s)

appendix = []
appendix.append(block("Table 1. Cohort and Network Processing Summary (current run)", t1))
appendix.append(block("Table 2. Backbone Audit (paper-like, top 10 by score)", t2_paper))
appendix.append(block("Table 3. Community size distribution (cluster_mod)", t3_paper))
appendix.append(block("Table 4. Associations Between Community Labels and Exogenous Metadata", t4_paper))
appendix.append(block("Table 7. IES-level summaries for works (2010+)", pd.read_csv(f"{PART6}/table7_ies_works_summary_paperlike.csv")))
appendix.append(block("Table 8. IES-level summaries for citations (2010+)", pd.read_csv(f"{PART6}/table8_ies_citations_summary_paperlike.csv")))
appendix.append(block("Table 9. Between-Cluster Differences (Kruskal-Wallis H Statistic)", t9_paper))
appendix.append(block("Table 13. Community quality beyond Silhouette (intra-PPGCI backbone)", t13))
appendix.append(block("Table 14. Conductance by community (lower is better)", t14.head(30)))
appendix.append(block("Table 15. Stability under edge bootstrap (Louvain vs. original labels)", t15))
appendix.append(block("Table 15b. Pairwise co-assignment persistence", t15p.head(60), max_rows=60))

appendix_txt = "\n".join(appendix)
out_txt = f"{PART6}/paper_tables_appendix_v2.txt"
with open(out_txt, "w", encoding="utf-8") as f:
    f.write(appendix_txt)

print("✅ PASSO 4 OK")
print("Appendix salvo em:", out_txt)
print("\nArquivos paper-like criados agora:")
new_files = [f for f in os.listdir(PART6) if "paperlike" in f or "appendix_v2" in f]
print(sorted(new_files))


✅ PASSO 4 OK
Appendix salvo em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/paper_tables_appendix_v2.txt

Arquivos paper-like criados agora:
['paper_tables_appendix_v2.txt', 'table13_quality_metrics_paperlike.csv', 'table14_conductance_by_cluster_paperlike.csv', 'table15_bootstrap_stability_paperlike.csv', 'table15_pairwise_persistence_paperlike.csv', 'table2_backbone_audit_paperlike.csv', 'table3_cluster_sizes_paperlike.csv', 'table4_associations_paperlike.csv', 'table7_ies_works_summary_paperlike.csv', 'table8_ies_citations_summary_paperlike.csv', 'table9_kruskal_by_cluster_paperlike.csv']


In [ ]:
import os
import pandas as pd

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"

# 1) Mostrar começo do TXT do apêndice
txt_path = f"{PART6}/paper_tables_appendix_v2.txt"
print("TXT exists?", os.path.exists(txt_path), "|", txt_path)

if os.path.exists(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    print("\n" + "="*80)
    print("APÊNDICE (primeiras ~2000 letras)")
    print("="*80)
    print(content[:2000])

    print("\n" + "="*80)
    print("APÊNDICE (últimas ~2000 letras)")
    print("="*80)
    print(content[-2000:])

# 2) Mostrar amostras das tabelas “paper-like” (head)
for fn in [
    "table2_backbone_audit_paperlike.csv",
    "table3_cluster_sizes_paperlike.csv",
    "table4_associations_paperlike.csv",
    "table9_kruskal_by_cluster_paperlike.csv",
]:
    p = f"{PART6}/{fn}"
    print("\n" + "-"*80)
    print("FILE:", fn, "| exists?", os.path.exists(p))
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(df.head(10).to_string(index=False))


TXT exists? True | /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/paper_tables_appendix_v2.txt

APÊNDICE (primeiras ~2000 letras)
Table 1. Cohort and Network Processing Summary (current run)
                                   Metric  Value
Works processed (rows in producoes_final)  38138
  Nodes in intra-PPGCI network (this run)    404
  Edges in intra-PPGCI network (this run)   1490
                          Communities (K)     16

Table 2. Backbone Audit (paper-like, top 10 by score)
 beta resolution     Q V_IES    p_IES H_norm  K score
0.750      1.500 0.704 0.621 2.1e-292  0.963 16 1.169
0.750      1.200 0.706 0.632 4.8e-287  0.980 15 1.152
1.000      1.200 0.698 0.645  <1e-300  0.967 15 1.146
1.000      1.500 0.699 0.636 2.1e-292  0.964 15 1.142
0.750      0.800 0.704 0.634 1.5e-269  0.967 14 1.120
0.750      0.600 0.699 0.630 2.0e-264  0.949 14 1.113
1.000      0.800 0.701 0.657 1.6e-275  0.967 13 1.094
1.000      0.600 0.692 0.611 5.9e-242  0.897 

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"
os.makedirs(PART6, exist_ok=True)

master_path = f"{BASE}/part4/master_node_table_v3.csv"
assert os.path.exists(master_path), f"Não achei master: {master_path}"

df = pd.read_csv(master_path).copy()

# -----------------------------
# 1) Preparar dados
# -----------------------------
# Targets
targets = ["citations_2010", "works_2010"]

# Features (numéricas)
num_feats = [
    "active_years_since2010",
    "papers_count_ppgci_internal",
    "log1p_strength",
    "log1p_degree",
    "avg_tie_strength",
    "degree",
    "strength",
    "cits_per_work",
]
num_feats = [c for c in num_feats if c in df.columns]

# Features (categóricas) — no-leak (não usa nenhum target)
cat_feats = []
for c in ["cluster_mod", "SG_ENTIDADE_ENSINO"]:
    if c in df.columns:
        cat_feats.append(c)

# Limpeza básica
for t in targets:
    df[t] = pd.to_numeric(df[t], errors="coerce").fillna(0.0)

for c in num_feats:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

for c in cat_feats:
    df[c] = df[c].astype("string").fillna("NA")

X = df[num_feats + cat_feats].copy()

# Preprocess: escala numérica + one-hot categórica
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_feats),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
    ],
    remainder="drop",
)

# Modelos
ridge = Pipeline([("prep", preprocess), ("model", Ridge(alpha=1.0, random_state=42))])
tree  = Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(random_state=42, max_depth=6))])

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# -----------------------------
# 2) Table 10 — desempenho preditivo
#    - Holdout (train/test)
#    - GroupKFold por IES (se houver)
# -----------------------------
rows = []

# HOLDOUT
for target in targets:
    y = df[target].values

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # Ridge
    ridge.fit(Xtr, ytr)
    pred = ridge.predict(Xte)
    rows.append({
        "Validation Method": "Holdout (25%)",
        "Target Variable": target,
        "Model": "Ridge",
        "R2": r2_score(yte, pred),
        "RMSE": rmse(yte, pred),
    })

    # Tree
    tree.fit(Xtr, ytr)
    pred = tree.predict(Xte)
    rows.append({
        "Validation Method": "Holdout (25%)",
        "Target Variable": target,
        "Model": "Tree",
        "R2": r2_score(yte, pred),
        "RMSE": rmse(yte, pred),
    })

# GROUPKFOLD (IES)
if "SG_ENTIDADE_ENSINO" in df.columns:
    groups = df["SG_ENTIDADE_ENSINO"].astype(str).values
    gkf = GroupKFold(n_splits=5)

    for target in targets:
        y = df[target].values

        r2s_ridge, rmses_ridge = [], []
        r2s_tree,  rmses_tree  = [], []

        for tr_idx, te_idx in gkf.split(X, y, groups=groups):
            Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
            ytr, yte = y[tr_idx], y[te_idx]

            # Ridge
            ridge.fit(Xtr, ytr)
            pred = ridge.predict(Xte)
            r2s_ridge.append(r2_score(yte, pred))
            rmses_ridge.append(rmse(yte, pred))

            # Tree
            tree.fit(Xtr, ytr)
            pred = tree.predict(Xte)
            r2s_tree.append(r2_score(yte, pred))
            rmses_tree.append(rmse(yte, pred))

        rows.append({
            "Validation Method": "GroupKFold (IES, 5-fold)",
            "Target Variable": target,
            "Model": "Ridge",
            "R2": float(np.mean(r2s_ridge)),
            "RMSE": float(np.mean(rmses_ridge)),
        })
        rows.append({
            "Validation Method": "GroupKFold (IES, 5-fold)",
            "Target Variable": target,
            "Model": "Tree",
            "R2": float(np.mean(r2s_tree)),
            "RMSE": float(np.mean(rmses_tree)),
        })

perf = pd.DataFrame(rows)

# salva Table 10
table10_path = f"{PART6}/table10_model_performance.csv"
perf.to_csv(table10_path, index=False)

print("✅ Table 10 salva:", table10_path)
print(perf)

# -----------------------------
# 3) Table 11 — Top Ridge coef (Citations)
# -----------------------------
def get_feature_names_from_ct(ct: ColumnTransformer):
    out = []
    for name, trans, cols in ct.transformers_:
        if name == "remainder":
            continue
        if hasattr(trans, "get_feature_names_out"):
            try:
                out.extend(list(trans.get_feature_names_out(cols)))
            except Exception:
                out.extend(list(cols))
        else:
            out.extend(list(cols))
    return out

def top_ridge_coefficients(pipeline: Pipeline, topk=15):
    model = pipeline.named_steps["model"]
    prep  = pipeline.named_steps["prep"]
    feat_names = get_feature_names_from_ct(prep)
    coefs = model.coef_.ravel()

    coef_df = pd.DataFrame({"Feature": feat_names, "Coefficient": coefs})
    coef_df["abs"] = coef_df["Coefficient"].abs()
    coef_df = coef_df.sort_values("abs", ascending=False).drop(columns=["abs"]).head(topk)
    return coef_df

# treino final em tudo (paper-style) e extrai coeficientes
ridge.fit(X, df["citations_2010"].values)
top_cit = top_ridge_coefficients(ridge, topk=15)

table11_path = f"{PART6}/table11_top_ridge_citations.csv"
top_cit.to_csv(table11_path, index=False)

print("\n✅ Table 11 salva:", table11_path)
print(top_cit.to_string(index=False))

# -----------------------------
# 4) Table 12 — Top Ridge coef (Works)
# -----------------------------
ridge.fit(X, df["works_2010"].values)
top_work = top_ridge_coefficients(ridge, topk=15)

table12_path = f"{PART6}/table12_top_ridge_works.csv"
top_work.to_csv(table12_path, index=False)

print("\n✅ Table 12 salva:", table12_path)
print(top_work.to_string(index=False))

print("\nArquivos gerados no PART6:")
print([f for f in os.listdir(PART6) if f.startswith("table10") or f.startswith("table11") or f.startswith("table12")])


✅ Table 10 salva: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/table10_model_performance.csv
          Validation Method Target Variable  Model        R2       RMSE
0             Holdout (25%)  citations_2010  Ridge  0.677179  57.104517
1             Holdout (25%)  citations_2010   Tree  0.609486  62.806948
2             Holdout (25%)      works_2010  Ridge  0.894164  14.691754
3             Holdout (25%)      works_2010   Tree  0.840744  18.022114
4  GroupKFold (IES, 5-fold)  citations_2010  Ridge  0.696091  64.034452
5  GroupKFold (IES, 5-fold)  citations_2010   Tree  0.699365  75.607842
6  GroupKFold (IES, 5-fold)      works_2010  Ridge  0.889984  17.600952
7  GroupKFold (IES, 5-fold)      works_2010   Tree  0.769058  25.372236

✅ Table 11 salva: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/table11_top_ridge_citations.csv
                     Feature  Coefficient
               cits_per_work   117.334780
 papers_cou

In [ ]:
import os, glob
import numpy as np
import pandas as pd

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"
INPUT = f"{BASE}/input"
os.makedirs(PART6, exist_ok=True)

master_path = f"{BASE}/part4/master_node_table_v3.csv"
assert os.path.exists(master_path), f"Não achei master: {master_path}"

df = pd.read_csv(master_path).copy()

# -----------------------------
# helpers
# -----------------------------
def detect_col(df_, candidates):
    lower_map = {c.lower(): c for c in df_.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def summarize_group(data, group_col, value_col, min_n=10, topk=12):
    d = data[[group_col, value_col]].dropna().copy()
    d[value_col] = pd.to_numeric(d[value_col], errors="coerce")
    d = d.dropna(subset=[value_col])
    # remove grupo NA / vazio
    d[group_col] = d[group_col].astype(str).replace({"nan":"NA","None":"NA"}).fillna("NA")
    d = d[d[group_col].str.strip() != ""]
    d = d[d[group_col] != "NA"]

    if d.empty:
        return pd.DataFrame()

    g = d.groupby(group_col)[value_col]
    out = pd.DataFrame({
        group_col: g.size().index,
        "n": g.size().values,
        "median": g.median().values,
        "mean": g.mean().values,
        "p75": g.quantile(0.75).values,
        "p90": g.quantile(0.90).values,
    })

    out = out.sort_values(["n","median"], ascending=[False, False]).reset_index(drop=True)

    # versão "paper" (como seu exemplo antigo costuma mostrar) — restringe a grupos com n>=min_n
    out_paper = out[out["n"] >= min_n].copy()
    out_paper = out_paper.head(topk).reset_index(drop=True)

    return out, out_paper

def try_build_uf_map_from_input(input_dir):
    """
    Procura em arquivos do input alguma tabela que tenha:
      - SG_ENTIDADE_ENSINO (ou equivalente)
      - UF (ou equivalente)
    e constrói mapeamento IES -> UF (pela moda).
    """
    if not os.path.isdir(input_dir):
        return None

    candidates = []
    candidates += glob.glob(os.path.join(input_dir, "*.xlsx"))
    candidates += glob.glob(os.path.join(input_dir, "*.csv"))

    # heurísticas de coluna
    ies_candidates = ["SG_ENTIDADE_ENSINO","ies","instituicao","instituicao_original","NM_ENTIDADE_ENSINO"]
    uf_candidates  = ["UF","uf","SG_UF","SG_UF_PROGRAMA","sg_uf_programa","sigla_uf"]

    for path in candidates:
        try:
            if path.lower().endswith(".xlsx"):
                tmp = pd.read_excel(path)
            else:
                tmp = pd.read_csv(path)

            ies_col = detect_col(tmp, ies_candidates)
            uf_col  = detect_col(tmp, uf_candidates)

            if ies_col and uf_col:
                # normaliza
                sub = tmp[[ies_col, uf_col]].dropna().copy()
                sub[ies_col] = sub[ies_col].astype(str).str.strip()
                sub[uf_col]  = sub[uf_col].astype(str).str.strip().str.upper()
                sub = sub[(sub[ies_col]!="") & (sub[uf_col]!="")]

                if sub.shape[0] >= 10:
                    # moda por IES
                    uf_map = (sub.groupby(ies_col)[uf_col]
                              .agg(lambda s: s.value_counts().idxmax())
                              .to_dict())
                    return uf_map, os.path.basename(path), ies_col, uf_col
        except Exception:
            continue

    return None

# -----------------------------
# 0) garantir colunas numéricas
# -----------------------------
for c in ["works_2010","citations_2010"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

assert "SG_ENTIDADE_ENSINO" in df.columns, "master precisa ter SG_ENTIDADE_ENSINO para Table 7–8."

# -----------------------------
# 1) Tabelas por IES (7 e 8) — sempre dá
# -----------------------------
ies_works_full, ies_works_paper = summarize_group(df, "SG_ENTIDADE_ENSINO", "works_2010", min_n=10, topk=12)
ies_cits_full,  ies_cits_paper  = summarize_group(df, "SG_ENTIDADE_ENSINO", "citations_2010", min_n=10, topk=12)

path7_full  = f"{PART6}/table7_ies_works_summary_FULL.csv"
path7_paper = f"{PART6}/table7_ies_works_summary.csv"
path8_full  = f"{PART6}/table8_ies_citations_summary_FULL.csv"
path8_paper = f"{PART6}/table8_ies_citations_summary.csv"

ies_works_full.to_csv(path7_full, index=False)
ies_works_paper.to_csv(path7_paper, index=False)
ies_cits_full.to_csv(path8_full, index=False)
ies_cits_paper.to_csv(path8_paper, index=False)

print("✅ Table 7 (IES-level works) salva:", path7_paper)
print(ies_works_paper.to_string(index=False))
print("\n✅ Table 8 (IES-level citations) salva:", path8_paper)
print(ies_cits_paper.to_string(index=False))

# -----------------------------
# 2) Tabelas por UF (5 e 6) — tenta recuperar UF automaticamente
# -----------------------------
uf_in_master = None
for cand in ["UF","uf","SG_UF","SG_UF_PROGRAMA"]:
    if cand in df.columns:
        uf_in_master = cand
        break

if uf_in_master is None:
    found = try_build_uf_map_from_input(INPUT)
    if found is not None:
        uf_map, src_file, ies_col_found, uf_col_found = found
        # aplica: master tem SG_ENTIDADE_ENSINO
        df["_UF"] = df["SG_ENTIDADE_ENSINO"].astype(str).map(uf_map)
        uf_in_master = "_UF"
        print(f"\n✅ UF recuperada via INPUT ({src_file}) usando [{ies_col_found}] -> [{uf_col_found}]")
    else:
        print("\n⚠️ Não encontrei UF no master e não consegui inferir UF a partir de BASE/input.")
        print("   Vou salvar apenas as tabelas por IES (7–8). Se você tiver um arquivo com IES->UF, eu plugo e gero 5–6.")

if uf_in_master is not None:
    uf_works_full, uf_works_paper = summarize_group(df.dropna(subset=[uf_in_master]), uf_in_master, "works_2010", min_n=10, topk=12)
    uf_cits_full,  uf_cits_paper  = summarize_group(df.dropna(subset=[uf_in_master]), uf_in_master, "citations_2010", min_n=10, topk=12)

    path5_full  = f"{PART6}/table5_uf_works_summary_FULL.csv"
    path5_paper = f"{PART6}/table5_uf_works_summary.csv"
    path6_full  = f"{PART6}/table6_uf_citations_summary_FULL.csv"
    path6_paper = f"{PART6}/table6_uf_citations_summary.csv"

    uf_works_full.to_csv(path5_full, index=False)
    uf_works_paper.to_csv(path5_paper, index=False)
    uf_cits_full.to_csv(path6_full, index=False)
    uf_cits_paper.to_csv(path6_paper, index=False)

    print("\n✅ Table 5 (UF-level works) salva:", path5_paper)
    print(uf_works_paper.to_string(index=False))
    print("\n✅ Table 6 (UF-level citations) salva:", path6_paper)
    print(uf_cits_paper.to_string(index=False))

print("\nArquivos gerados/atualizados no PART6 (Table 5–8):")
print([f for f in os.listdir(PART6) if f.startswith("table5") or f.startswith("table6") or f.startswith("table7") or f.startswith("table8")])


✅ Table 7 (IES-level works) salva: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/table7_ies_works_summary.csv
SG_ENTIDADE_ENSINO  n  median       mean    p75   p90
              UFMG 39    27.0  57.051282  69.00 134.6
         UNESP-MAR 32    87.0 114.375000 160.50 228.3
            UNIRIO 31    16.0  23.193548  27.50  35.0
             UDESC 21    36.0  41.190476  56.00  66.0
           UFPB-JP 20    71.5  88.050000 114.25 157.5
              UFSC 20    66.0  78.200000  85.00 163.9
               UNB 20    32.5  42.450000  69.25  76.9
              UFBA 20    15.5  29.750000  34.75  74.0
               USP 19    31.0  33.473684  54.50  60.6
               UFF 18    24.5  35.388889  47.25  73.5
               UEL 17    27.0  47.294118  51.00  74.4
              UFRJ 16    50.5  46.375000  63.75  77.5

✅ Table 8 (IES-level citations) salva: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/table8_ies_citations_summary.csv
SG_

In [ ]:
import os
import pandas as pd
from datetime import datetime

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"
assert os.path.isdir(PART6), f"PART6 não existe: {PART6}"

def pick_file(stems):
    """
    Escolhe o melhor arquivo existente no PART6, na ordem:
      1) *_paperlike.csv
      2) *.csv
    """
    for stem in stems:
        p1 = os.path.join(PART6, f"{stem}_paperlike.csv")
        p2 = os.path.join(PART6, f"{stem}.csv")
        if os.path.exists(p1):
            return p1
        if os.path.exists(p2):
            return p2
    return None

def read_csv_safe(path):
    if path is None:
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        return f"ERROR reading {os.path.basename(path)}: {e}"

def df_to_text(df, max_rows=80):
    if df is None:
        return "❌ (arquivo não encontrado)\n"
    if isinstance(df, str):
        return df + "\n"
    if df.empty:
        return "(tabela vazia)\n"
    # imprime muita coisa, mas com limite para não explodir
    if df.shape[0] > max_rows:
        head = df.head(max_rows).to_string(index=False)
        return head + f"\n... ({df.shape[0]-max_rows} linhas omitidas)\n"
    return df.to_string(index=False) + "\n"

# ----------------------------
# Mapeamento: quais tabelas entram no apêndice
# ----------------------------
tables = [
    ("Table 1. Cohort and Network Processing Summary", ["table1_processing_summary"]),
    ("Table 2. Backbone Audit Across β Values",        ["table2_backbone_audit_paperlike", "table2_backbone_audit_top10", "table2_backbone_audit_full"]),
    ("Table 3. Community size distribution (cluster_mod)", ["table3_cluster_sizes"]),
    ("Table 4. Associations Between Community Labels and Exogenous Metadata", ["table4_associations"]),
    ("Table 5. UF-level summaries for works (2010+)",  ["table5_uf_works_summary"]),
    ("Table 6. UF-level summaries for citations (2010+)", ["table6_uf_citations_summary"]),
    ("Table 7. IES-level summaries for works (2010+)", ["table7_ies_works_summary"]),
    ("Table 8. IES-level summaries for citations (2010+)", ["table8_ies_citations_summary"]),
    ("Table 9. Between-Cluster Differences (Kruskal–Wallis H Statistic)", ["table9_kruskal_by_cluster"]),
    ("Table 10. Predictive model performance across validation strategies", ["table10_model_performance"]),
    ("Table 11. Top Ridge Coefficients — Predicting Citations (2010+, no-leak)", ["table11_top_ridge_citations"]),
    ("Table 12. Top Ridge Coefficients — Predicting Works (2010+, no-leak)", ["table12_top_ridge_works"]),
    ("Table 13. Community quality beyond Silhouette (intra-PPGCI backbone)", ["table13_quality_metrics"]),
    ("Table 14. Conductance by community (lower is better)", ["table14_conductance_by_cluster"]),
    ("Table 15. Stability under edge bootstrap (Louvain vs reference)", ["table15_bootstrap_stability"]),
    ("Table 16. Pairwise co-assignment persistence", ["table15_pairwise_persistence"]),  # mantive 16 para não repetir “15”
]

# ----------------------------
# Gera apêndice
# ----------------------------
lines = []
lines.append("PAPER TABLES APPENDIX (v3)")
lines.append("=" * 80)
lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"BASE:  {BASE}")
lines.append(f"PART6: {PART6}")
lines.append("=" * 80 + "\n")

missing = []

for title, stems in tables:
    # tenta localizar arquivo
    path = None
    for stem in stems:
        # stem pode vir já com "_paperlike" (por compatibilidade com sua lista)
        # então testamos direto e também a variação sem isso
        if stem.endswith("_paperlike"):
            p = os.path.join(PART6, f"{stem}.csv")
            if os.path.exists(p):
                path = p
                break
            stem2 = stem.replace("_paperlike","")
            p2 = os.path.join(PART6, f"{stem2}_paperlike.csv")
            if os.path.exists(p2):
                path = p2
                break
            p3 = os.path.join(PART6, f"{stem2}.csv")
            if os.path.exists(p3):
                path = p3
                break
        else:
            p = os.path.join(PART6, f"{stem}_paperlike.csv")
            p2 = os.path.join(PART6, f"{stem}.csv")
            if os.path.exists(p):
                path = p
                break
            if os.path.exists(p2):
                path = p2
                break

    df = read_csv_safe(path)

    lines.append(title)
    lines.append("-" * len(title))
    if path:
        lines.append(f"[source: {os.path.basename(path)}]")
    else:
        missing.append(title)
        lines.append("[source: ❌ missing]")
    lines.append(df_to_text(df))
    lines.append("")

appendix_path = os.path.join(PART6, "paper_tables_appendix_v3.txt")
with open(appendix_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("✅ PASSO 7 OK")
print("Appendix salvo em:", appendix_path)

# ----------------------------
# Imprime na tela (preview grande)
# ----------------------------
print("\n" + "="*80)
print("PREVIEW (primeiras ~350 linhas)")
print("="*80)
preview = "\n".join(lines[:350])
print(preview)

if missing:
    print("\n" + "="*80)
    print("⚠️ Tabelas faltando no PART6:")
    for t in missing:
        print(" -", t)
else:
    print("\n✅ Nenhuma tabela faltando — apêndice completo.")


✅ PASSO 7 OK
Appendix salvo em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6/paper_tables_appendix_v3.txt

PREVIEW (primeiras ~350 linhas)
PAPER TABLES APPENDIX (v3)
Generated: 2026-02-03 15:48:12
BASE:  /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407
PART6: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6

Table 1. Cohort and Network Processing Summary
----------------------------------------------
[source: table1_processing_summary.csv]
                                   Metric  Value
Works processed (rows in producoes_final)  38138
  Nodes in intra-PPGCI network (this run)    404
  Edges in intra-PPGCI network (this run)   1490
                          Communities (K)     16


Table 2. Backbone Audit Across β Values
---------------------------------------
[source: table2_backbone_audit_paperlike.csv]
 beta  resolution     Q  V_IES    p_IES  H_norm  K  score
 0.75         1.5 0.704  0.621 2.1e-2

In [ ]:
import os, shutil
import pandas as pd
from datetime import datetime

BASE  = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
PART6 = f"{BASE}/part6"
OUT   = f"{BASE}/paper_outputs_v2"

assert os.path.isdir(PART6), f"PART6 não existe: {PART6}"
os.makedirs(OUT, exist_ok=True)

def best_source(stem):
    """
    Preferência:
      1) stem_paperlike.csv
      2) stem.csv
    """
    p1 = os.path.join(PART6, f"{stem}_paperlike.csv")
    p2 = os.path.join(PART6, f"{stem}.csv")
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

# Mapeamento final (nome de saída -> stem origem)
# (Incluí Table 16 para persistência, para não duplicar "15")
table_map = [
    ("Table1_processing_summary.csv",           "table1_processing_summary"),
    ("Table2_backbone_audit.csv",               "table2_backbone_audit"),   # fallback abaixo
    ("Table3_cluster_sizes.csv",                "table3_cluster_sizes"),
    ("Table4_associations.csv",                 "table4_associations"),
    ("Table5_uf_works_summary.csv",             "table5_uf_works_summary"),
    ("Table6_uf_citations_summary.csv",         "table6_uf_citations_summary"),
    ("Table7_ies_works_summary.csv",            "table7_ies_works_summary"),
    ("Table8_ies_citations_summary.csv",        "table8_ies_citations_summary"),
    ("Table9_kruskal_by_cluster.csv",           "table9_kruskal_by_cluster"),
    ("Table10_model_performance.csv",           "table10_model_performance"),
    ("Table11_top_ridge_citations.csv",         "table11_top_ridge_citations"),
    ("Table12_top_ridge_works.csv",             "table12_top_ridge_works"),
    ("Table13_quality_metrics.csv",             "table13_quality_metrics"),
    ("Table14_conductance_by_cluster.csv",      "table14_conductance_by_cluster"),
    ("Table15_bootstrap_stability.csv",         "table15_bootstrap_stability"),
    ("Table16_pairwise_persistence.csv",        "table15_pairwise_persistence"),
]

manifest = []

def copy_one(src, dst_name, kind="table"):
    dst = os.path.join(OUT, dst_name)
    shutil.copy2(src, dst)
    manifest.append({
        "kind": kind,
        "output_file": dst_name,
        "source_file": os.path.relpath(src, BASE),
        "bytes": os.path.getsize(dst),
    })

# 1) Copiar appendix
appendix = os.path.join(PART6, "paper_tables_appendix_v3.txt")
if os.path.exists(appendix):
    copy_one(appendix, "paper_tables_appendix_v3.txt", kind="appendix")
else:
    print("⚠️ Não achei paper_tables_appendix_v3.txt em part6 (rode o passo 7 antes).")

# 2) Table 2 tem variações — escolher a melhor disponível
# tenta: table2_backbone_audit_paperlike.csv, depois top10/full
t2_candidates = [
    os.path.join(PART6, "table2_backbone_audit_paperlike.csv"),
    os.path.join(PART6, "table2_backbone_audit_full.csv"),
    os.path.join(PART6, "table2_backbone_audit_top10.csv"),
    os.path.join(PART6, "table2_backbone_audit_full_paperlike.csv"),
    os.path.join(PART6, "table2_backbone_audit_top10_paperlike.csv"),
]
t2_src = None
for p in t2_candidates:
    if os.path.exists(p):
        t2_src = p
        break

# 3) Copiar todas as tabelas
missing = []
for out_name, stem in table_map:
    if stem == "table2_backbone_audit":
        if t2_src:
            copy_one(t2_src, out_name)
        else:
            missing.append(out_name)
        continue

    src = best_source(stem)
    if src:
        copy_one(src, out_name)
    else:
        missing.append(out_name)

# 4) Manifest + README curtinho
manifest_df = pd.DataFrame(manifest).sort_values(["kind","output_file"])
manifest_path = os.path.join(OUT, "MANIFEST_paper_outputs_v2.csv")
manifest_df.to_csv(manifest_path, index=False)

readme_path = os.path.join(OUT, "README_paper_outputs_v2.txt")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write("paper_outputs_v2\n")
    f.write("="*60 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"BASE:  {BASE}\n")
    f.write(f"FROM:  {PART6}\n")
    f.write(f"TO:    {OUT}\n\n")
    f.write("Contents:\n")
    for _, r in manifest_df.iterrows():
        f.write(f" - {r['output_file']}  (src: {r['source_file']})\n")
    if missing:
        f.write("\nMissing outputs:\n")
        for m in missing:
            f.write(f" - {m}\n")

print("✅ PASSO 8 OK")
print("Saída final em:", OUT)
print("Arquivos (amostra):", sorted(os.listdir(OUT))[:25])
print("Manifest:", manifest_path)
print("README :", readme_path)

if missing:
    print("\n⚠️ Faltaram estes arquivos (não encontrei no PART6):")
    for m in missing:
        print(" -", m)
else:
    print("\n✅ Pacote completo — nada faltando.")


✅ PASSO 8 OK
Saída final em: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2
Arquivos (amostra): ['MANIFEST_paper_outputs_v2.csv', 'README_paper_outputs_v2.txt', 'Table10_model_performance.csv', 'Table11_top_ridge_citations.csv', 'Table12_top_ridge_works.csv', 'Table13_quality_metrics.csv', 'Table14_conductance_by_cluster.csv', 'Table15_bootstrap_stability.csv', 'Table16_pairwise_persistence.csv', 'Table1_processing_summary.csv', 'Table2_backbone_audit.csv', 'Table3_cluster_sizes.csv', 'Table4_associations.csv', 'Table5_uf_works_summary.csv', 'Table6_uf_citations_summary.csv', 'Table7_ies_works_summary.csv', 'Table8_ies_citations_summary.csv', 'Table9_kruskal_by_cluster.csv', 'paper_tables_appendix_v3.txt']
Manifest: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2/MANIFEST_paper_outputs_v2.csv
README : /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2/README_paper

In [ ]:
import os, zipfile
import pandas as pd
from datetime import datetime

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
OUT  = f"{BASE}/paper_outputs_v2"
assert os.path.isdir(OUT), f"paper_outputs_v2 não existe: {OUT}"

def load_csv(path):
    return pd.read_csv(path)

def df_to_text(df, max_rows=200):
    # impressão limpa (sem índice)
    if df.shape[0] > max_rows:
        df = df.head(max_rows)
    return df.to_string(index=False)

order = [
    ("Table 1. Cohort and Network Processing Summary", "Table1_processing_summary.csv"),
    ("Table 2. Backbone Audit Across β Values (selected config highlighted in your narrative)", "Table2_backbone_audit.csv"),
    ("Table 3. Community size distribution (cluster_mod)", "Table3_cluster_sizes.csv"),
    ("Table 4. Associations Between Community Labels and Exogenous Metadata", "Table4_associations.csv"),
    ("Table 5. UF-level summaries for works (2010+)", "Table5_uf_works_summary.csv"),
    ("Table 6. UF-level summaries for citations (2010+)", "Table6_uf_citations_summary.csv"),
    ("Table 7. IES-level summaries for works (2010+)", "Table7_ies_works_summary.csv"),
    ("Table 8. IES-level summaries for citations (2010+)", "Table8_ies_citations_summary.csv"),
    ("Table 9. Between-Cluster Differences (Kruskal–Wallis)", "Table9_kruskal_by_cluster.csv"),
    ("Table 10. Predictive model performance across validation strategies", "Table10_model_performance.csv"),
    ("Table 11. Top Ridge Coefficients — Predicting Citations (2010+, no-leak)", "Table11_top_ridge_citations.csv"),
    ("Table 12. Top Ridge Coefficients — Predicting Works (2010+, no-leak)", "Table12_top_ridge_works.csv"),
    ("Table 13. Community quality beyond Silhouette (intra-PPGCI backbone)", "Table13_quality_metrics.csv"),
    ("Table 14. Conductance by community (lower is better)", "Table14_conductance_by_cluster.csv"),
    ("Table 15. Stability under edge bootstrap (Louvain vs reference)", "Table15_bootstrap_stability.csv"),
    ("Table 16. Pairwise co-assignment persistence", "Table16_pairwise_persistence.csv"),
]

pack_txt = os.path.join(OUT, "paper_tables_pack_v1.txt")

with open(pack_txt, "w", encoding="utf-8") as f:
    f.write("PPGCI intra-field lab — Paper Tables Pack (v1)\n")
    f.write("="*72 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Source folder: {OUT}\n\n")

    # inclui appendix já existente (se houver)
    appendix = os.path.join(OUT, "paper_tables_appendix_v3.txt")
    if os.path.exists(appendix):
        f.write("APPENDIX (as generated)\n")
        f.write("-"*72 + "\n")
        f.write(open(appendix, "r", encoding="utf-8").read())
        f.write("\n\n")

    # inclui todas as tabelas
    for title, fn in order:
        path = os.path.join(OUT, fn)
        if not os.path.exists(path):
            f.write(f"{title}\n")
            f.write("-"*72 + "\n")
            f.write(f"[MISSING FILE] {fn}\n\n")
            continue

        df = load_csv(path)
        f.write(f"{title}\n")
        f.write("-"*72 + "\n")
        f.write(df_to_text(df))
        f.write("\n\n")

print("✅ PASSO 9A OK — pack gerado:")
print(" -", pack_txt)

# ------------------------------------------------------------
# ZIP final do paper_outputs_v2 (pra download/compartilhar)
# ------------------------------------------------------------
zip_path = os.path.join(BASE, "paper_outputs_v2.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(OUT):
        for file in files:
            full = os.path.join(root, file)
            rel  = os.path.relpath(full, BASE)  # mantém estrutura a partir do BASE
            z.write(full, rel)

print("\n✅ PASSO 9B OK — ZIP criado:")
print(" -", zip_path)


✅ PASSO 9A OK — pack gerado:
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2/paper_tables_pack_v1.txt

✅ PASSO 9B OK — ZIP criado:
 - /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2.zip


In [ ]:
import os, glob, pandas as pd

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
OUT  = f"{BASE}/paper_outputs_v2"

assert os.path.exists(OUT), f"Não achei: {OUT}"
print("OK:", OUT)

print("\nArquivos no paper_outputs_v2 (amostra):")
files = sorted(os.listdir(OUT))
print(files[:30], "\n... total:", len(files))


OK: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2

Arquivos no paper_outputs_v2 (amostra):
['MANIFEST_paper_outputs_v2.csv', 'README_paper_outputs_v2.txt', 'Table10_model_performance.csv', 'Table11_top_ridge_citations.csv', 'Table12_top_ridge_works.csv', 'Table13_quality_metrics.csv', 'Table14_conductance_by_cluster.csv', 'Table15_bootstrap_stability.csv', 'Table16_pairwise_persistence.csv', 'Table1_processing_summary.csv', 'Table2_backbone_audit.csv', 'Table3_cluster_sizes.csv', 'Table4_associations.csv', 'Table5_uf_works_summary.csv', 'Table6_uf_citations_summary.csv', 'Table7_ies_works_summary.csv', 'Table8_ies_citations_summary.csv', 'Table9_kruskal_by_cluster.csv', 'paper_tables_appendix_v3.txt', 'paper_tables_pack_v1.txt'] 
... total: 20


In [ ]:
import pandas as pd, os

def show_csv(name, n=50):
    path = os.path.join(OUT, name)
    print("\n" + "="*90)
    print(name)
    print("="*90)
    df = pd.read_csv(path)
    display(df.head(n))
    return df

# Principais tabelas (as mesmas do manifest)
to_show = [
    "Table1_processing_summary.csv",
    "Table2_backbone_audit.csv",
    "Table3_cluster_sizes.csv",
    "Table4_associations.csv",
    "Table5_uf_works_summary.csv",
    "Table6_uf_citations_summary.csv",
    "Table7_ies_works_summary.csv",
    "Table8_ies_citations_summary.csv",
    "Table9_kruskal_by_cluster.csv",
    "Table10_model_performance.csv",
    "Table11_top_ridge_citations.csv",
    "Table12_top_ridge_works.csv",
    "Table13_quality_metrics.csv",
    "Table14_conductance_by_cluster.csv",
    "Table15_bootstrap_stability.csv",
    "Table16_pairwise_persistence.csv",
]

for f in to_show:
    if os.path.exists(os.path.join(OUT, f)):
        show_csv(f, n=80)
    else:
        print("⚠️ faltando:", f)



Table1_processing_summary.csv


,Metric,Value
0,Works processed (rows in producoes_final),38138
1,Nodes in intra-PPGCI network (this run),404
2,Edges in intra-PPGCI network (this run),1490
3,Communities (K),16



Table2_backbone_audit.csv


,beta,resolution,Q,V_IES,p_IES,H_norm,K,score
0,0.75,1.5,0.704,0.621,2.1e-292,0.963,16,1.169
1,0.75,1.2,0.706,0.632,4.8e-287,0.980,15,1.152
2,1.00,1.2,0.698,0.645,<1e-300,0.967,15,1.146
3,1.00,1.5,0.699,0.636,2.1e-292,0.964,15,1.142
4,0.75,0.8,0.704,0.634,1.5e-269,0.967,14,1.120
5,0.75,0.6,0.699,0.630,2.0e-264,0.949,14,1.113
6,1.00,0.8,0.701,0.657,1.6e-275,0.967,13,1.094
7,1.00,0.6,0.692,0.611,5.9e-242,0.897,14,1.084
8,1.00,1.0,0.702,0.636,3.9e-231,0.931,12,1.045
9,0.75,1.0,0.708,0.654,6.6e-228,0.958,11,1.039



Table3_cluster_sizes.csv


...

,cluster_mod,size,volume,cut,conductance
0,0,42,1489.0,341.0,0.229013
1,1,19,2031.0,449.0,0.221073
2,2,24,1007.0,113.0,0.112214
3,3,19,571.0,115.0,0.201401
4,4,34,1532.0,284.0,0.185379
5,5,27,2580.0,700.0,0.271318
6,6,29,3447.0,459.0,0.133159
7,7,27,2310.0,690.0,0.298701
8,8,28,2077.0,577.0,0.277805
9,9,19,1616.0,526.0,0.325495



Table15_bootstrap_stability.csv


,bootstrap,nmi_to_reference,ari_to_reference
0,0,0.574968,0.278160
1,1,0.610513,0.345330
2,2,0.628134,0.369918
3,3,0.630771,0.355619
4,4,0.598123,0.311783
5,5,0.585075,0.346460
6,6,0.617267,0.335252
7,7,0.643248,0.405550
8,8,0.641245,0.371288
9,9,0.625694,0.388648



Table16_pairwise_persistence.csv


,key,value,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,global_mean,0.344071,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,by_community,NaN,"{'n_pairs': 861, 'mean_persistence': 0.2076190...","{'n_pairs': 171, 'mean_persistence': 0.3922807...","{'n_pairs': 276, 'mean_persistence': 0.5733333...","{'n_pairs': 171, 'mean_persistence': 0.3375438...","{'n_pairs': 561, 'mean_persistence': 0.4843850...","{'n_pairs': 351, 'mean_persistence': 0.4020512...","{'n_pairs': 406, 'mean_persistence': 0.5439901...","{'n_pairs': 351, 'mean_persistence': 0.2990313...","{'n_pairs': 378, 'mean_persistence': 0.2641269...","{'n_pairs': 171, 'mean_persistence': 0.4334502...","{'n_pairs': 946, 'mean_persistence': 0.2564270...","{'n_pairs': 741, 'mean_persistence': 0.1912820...","{'n_pairs': 78, 'mean_persistence': 0.29282051...","{'n_pairs': 300, 'mean_persistence': 0.5914}","{'n_pairs': 66, 'mean_persistence': 0.68151515...","{'n_pairs': 3, 'mean_persistence': 0.760000000..."


In [ ]:
import os

appendix = os.path.join(OUT, "paper_tables_appendix_v3.txt")
assert os.path.exists(appendix), f"Não achei: {appendix}"

print("\n" + "="*90)
print("paper_tables_appendix_v3.txt (início)")
print("="*90)

with open(appendix, "r", encoding="utf-8") as f:
    txt = f.read()

print(txt[:4000])  # mostra os primeiros 4000 caracteres
print("\n... [TRUNCADO] ...\n")



paper_tables_appendix_v3.txt (início)
PAPER TABLES APPENDIX (v3)
Generated: 2026-02-03 15:48:12
BASE:  /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407
PART6: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/part6

Table 1. Cohort and Network Processing Summary
----------------------------------------------
[source: table1_processing_summary.csv]
                                   Metric  Value
Works processed (rows in producoes_final)  38138
  Nodes in intra-PPGCI network (this run)    404
  Edges in intra-PPGCI network (this run)   1490
                          Communities (K)     16


Table 2. Backbone Audit Across β Values
---------------------------------------
[source: table2_backbone_audit_paperlike.csv]
 beta  resolution     Q  V_IES    p_IES  H_norm  K  score
 0.75         1.5 0.704  0.621 2.1e-292   0.963 16  1.169
 0.75         1.2 0.706  0.632 4.8e-287   0.980 15  1.152
 1.00         1.2 0.698  0.645  <1e-300   0.967 15  1.1

In [ ]:
import shutil, os

zip_path = shutil.make_archive(
    base_name=os.path.join(BASE, "paper_outputs_v2_PACKAGE"),
    format="zip",
    root_dir=OUT
)

print("✅ ZIP gerado:", zip_path)


✅ ZIP gerado: /content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2_PACKAGE.zip


In [ ]:
!pip install -q python-docx
import docx
print("python-docx version:", docx.__version__)
from docx import Document
import pandas as pd
import os

BASE = "/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407"
OUT  = f"{BASE}/paper_outputs_v2"

docx_out = os.path.join(BASE, "paper_outputs_v2_ALL_TABLES.docx")

doc = Document()
doc.add_heading("PPGCI intra-field lab — Tabelas (paper_outputs_v2)", level=1)

def add_df_as_table(doc, title, df):
    doc.add_heading(title.replace(".csv",""), level=2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    hdr = table.rows[0].cells
    for j, col in enumerate(df.columns):
        hdr[j].text = str(col)

    for _, row in df.iterrows():
        cells = table.add_row().cells
        for j, col in enumerate(df.columns):
            v = row[col]
            cells[j].text = "" if pd.isna(v) else str(v)

    doc.add_paragraph("")

tables = [
    "Table1_processing_summary.csv",
    "Table2_backbone_audit.csv",
    "Table3_cluster_sizes.csv",
    "Table4_associations.csv",
    "Table5_uf_works_summary.csv",
    "Table6_uf_citations_summary.csv",
    "Table7_ies_works_summary.csv",
    "Table8_ies_citations_summary.csv",
    "Table9_kruskal_by_cluster.csv",
    "Table10_model_performance.csv",
    "Table11_top_ridge_citations.csv",
    "Table12_top_ridge_works.csv",
    "Table13_quality_metrics.csv",
    "Table14_conductance_by_cluster.csv",
    "Table15_bootstrap_stability.csv",
    "Table16_pairwise_persistence.csv",
]

for fname in tables:
    path = os.path.join(OUT, fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        add_df_as_table(doc, fname, df)

# Appendix
appendix = os.path.join(OUT, "paper_tables_appendix_v3.txt")
if os.path.exists(appendix):
    doc.add_heading("Appendix — paper_tables_appendix_v3.txt", level=2)
    with open(appendix, "r", encoding="utf-8") as f:
        doc.add_paragraph(f.read())

doc.save(docx_out)

print("✅ DOCX gerado com sucesso:")
print(docx_out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.0 MB/s eta 0:00:00
python-docx version: 1.2.0
✅ DOCX gerado com sucesso:
/content/drive/MyDrive/PPGCI_intra_field_lab_v2/runs/run_20260202_175407/paper_outputs_v2_ALL_TABLES.docx
